# DATA209 — Advanced Exploratory Data Analysis
## Laboratory Manual

**Vidyashilp University · School of Engineering and Technology**
BTech Hons. (Data Science), Semester III · 4 credits (2:0:4) · 60 laboratory hours

---

### Primary dataset

**Online Shoppers Purchasing Intention** — 12,330 browsing sessions x 18 columns
(UCI Machine Learning Repository). One row per session on an e-commerce site over twelve
months, with a boolean `Revenue` target marking sessions that ended in a purchase.

It is used for every practical in this manual. Three secondary files appear where the
modality demands it:

| File | Used in | Why |
|---|---|---|
| `Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv` | P3-4 | free text for word frequency |
| `HistoricalPrices.csv` | P3-4 | a genuine daily time series |
| `1.png`, `1.wav` | P5-6 | image and audio representation |
| `pima.csv` | P17-18 | real disguised missingness |

Every cell falls back gracefully if a secondary file is absent, so the manual always runs.

### How to use this manual

1. Put the CSV files in the same folder as this notebook, or set `DATA_DIR` in the setup cell.
2. Run the **setup cell** first, every session.
3. Work through the practicals **in order** — later sessions use objects built in earlier ones.
   If you start in the middle, use *Cell -> Run All Above* first.
4. Every practical ends with a **deliverable**. Read it before you start.

### The rule that carries the most marks

> Every figure and every table needs one sentence underneath saying what it shows about
> the problem. A chart without an interpretation earns no marks in this course.

### Sessions in this manual

| Session | Week | Module | Title |
|---|---|---|---|
| P1-2 | Week 1 | 1 | Load and summarise |
| P3-4 | Week 2 | 1 | Type handling, text and time series |
| P5-6 | Week 3 | 1 | Unstructured data — image and audio |
| P7-8 | Week 4 | 2 | Univariate EDA |
| P9-10 | Week 5 | 2 | Bivariate analysis and hypothesis generation |
| P11-12 | Week 6 | 2 | Grouping — K-means, silhouette and pair plots |
| P15-16 | Week 8 | 3 | Data quality audit |
| P17-18 | Week 9 | 3 | Missing values and imputation |
| P19-20 | Week 10 | 3 | Outlier detection and treatment |
| P23-24 | Week 12 | 4 | Transformation and scaling |
| P25-26 | Week 13 | 4 | Feature engineering and selection |
| P27-28 | Week 14 | 4 | PCA — implementation and interpretation |
| P29-30 | Week 15 | 4 | Full pipeline — industry-style notebook |

*P13-14 and P21-22 are assignment sessions and have no new lab content; see the assignment
briefs on the course website.*

---


## Setup

Run this cell first, every session.

In [ ]:
# ============================================================
# DATA209 — Advanced Exploratory Data Analysis
# Lab manual: shared setup. Run this cell first, every session.
# ============================================================
import warnings, os, math, textwrap
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.dpi"]  = 110
plt.rcParams["figure.figsize"] = (9, 4)
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# ------------------------------------------------------------------
# EDIT THIS: the folder holding the course CSVs on your machine.
# Keep the data next to this notebook and "." will just work.
# ------------------------------------------------------------------
DATA_DIR = "."

def find(filename, folder=None):
    """Locate a course file, searching DATA_DIR recursively. Returns None if absent."""
    root = folder or DATA_DIR
    direct = os.path.join(root, filename)
    if os.path.exists(direct):
        return direct
    for dirpath, _, files in os.walk(root):
        if filename in files:
            return os.path.join(dirpath, filename)
    return None

print("pandas", pd.__version__, "| numpy", np.__version__)
print("Data folder:", os.path.abspath(DATA_DIR))

---

# P1-2 · Load and summarise

**Week 1 · Module 1 · CO1**

**Objective.** Frame the problem, load the dataset, separate dependent from independent variables, and describe every numeric column by centre, spread and shape.


### Problem statement

An online retailer wants to understand **which browsing sessions end in a purchase**, so that
marketing spend can be directed at the sessions most likely to convert.

The data is one row per browsing session on an e-commerce site over a twelve-month period.
Before any predictive model is considered, we must establish what the data contains, whether
it can support the question, and what its variables actually look like.

**Dataset — Online Shoppers Purchasing Intention** (UCI Machine Learning Repository)
12,330 sessions x 18 columns; 10 numeric, 8 categorical or boolean.
This is the primary dataset for the whole lab manual.

In [ ]:
# ---- Load the dataset -------------------------------------------------
path = find("online_shoppers_intention.csv")
if path is None:
    raise FileNotFoundError(
        "online_shoppers_intention.csv not found. Set DATA_DIR in the setup cell, "
        "or download it from the UCI repository."
    )

df = pd.read_csv(path)
print("Loaded:", path)
print("Shape :", df.shape, "->", f"{df.shape[0]:,} rows x {df.shape[1]} columns")
df.head()

In [ ]:
# ---- Structure before statistics --------------------------------------
# Never compute a mean before you know the column's type is right.
structure = pd.DataFrame({
    "dtype"   : df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "nulls"   : df.isna().sum(),
    "unique"  : df.nunique(),
    "example" : df.iloc[0],
})
print(structure.to_string())
print("\nMemory:", round(df.memory_usage(deep=True).sum() / 1e6, 2), "MB")

### Dependent and independent variables

The **dependent variable** (target) is what we want to explain or predict.
The **independent variables** (features) are everything we might explain it with.

Getting this split right matters: the target must never be used as an input, and any variable
that would not be known at the moment of prediction must be excluded as well.

In [ ]:
# ---- Dependent vs independent ------------------------------------------
TARGET = "Revenue"          # True if the session ended in a purchase

y = df[TARGET]
X = df.drop(columns=[TARGET])

numeric_cols     = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = [c for c in X.columns if c not in numeric_cols]

print(f"Dependent variable : {TARGET}  (dtype {y.dtype})")
print(f"Independent        : {X.shape[1]} variables")
print(f"  numeric      ({len(numeric_cols):2}): {numeric_cols}")
print(f"  categorical  ({len(categorical_cols):2}): {categorical_cols}")

print("\nTarget balance:")
print((y.value_counts(normalize=True) * 100).round(2).to_string())
print("\n-> 15.5% positive. Any model predicting 'no purchase' every time is 84.5% accurate.")
print("   Accuracy is therefore not an honest metric for this problem.")

### Mean vs. median

The mean is the balance point of the distribution; the median is the value that splits it in half.
They agree on symmetric data and diverge on skewed data. **The size of the gap is the diagnostic**:
a mean far above the median means a long right tail pulling the average up.

In [ ]:
# ---- Mean vs median ----------------------------------------------------
centre = pd.DataFrame({
    "mean"  : df[numeric_cols].mean(),
    "median": df[numeric_cols].median(),
})
centre["difference"]  = centre["mean"] - centre["median"]
# ratio guards against divide-by-zero on columns whose median is 0
centre["mean/median"] = np.where(centre["median"] != 0,
                                 centre["mean"] / centre["median"], np.nan)
centre = centre.sort_values("difference", ascending=False)
print(centre.to_string())

print("\nInterpretation")
print("- Where median = 0 the majority of sessions never touched that page type at all.")
print("- ProductRelated_Duration: mean far exceeds median -> strong right skew.")
print("- Report the median for these columns; the mean describes no typical session.")

### Variance vs. IQR

Variance and standard deviation measure spread around the *mean*, so they inherit its sensitivity
to extreme values. The IQR measures the width of the middle 50% and ignores the tails entirely.

Pair them correctly: **mean with standard deviation**, or **median with IQR**. Never mix.

In [ ]:
# ---- Variance vs IQR ---------------------------------------------------
q1 = df[numeric_cols].quantile(0.25)
q3 = df[numeric_cols].quantile(0.75)

spread = pd.DataFrame({
    "variance": df[numeric_cols].var(),
    "std"     : df[numeric_cols].std(),
    "IQR"     : q3 - q1,
    "range"   : df[numeric_cols].max() - df[numeric_cols].min(),
})
spread["std/IQR"] = np.where(spread["IQR"] != 0, spread["std"] / spread["IQR"], np.nan)
print(spread.sort_values("std/IQR", ascending=False).to_string())

print("\nInterpretation")
print("- std/IQR far above 1 means the tails are inflating the standard deviation.")
print("- Those columns need a robust summary (median + IQR) and, later, a transformation.")

### Skewness and distribution shape

Skewness quantifies horizontal asymmetry; kurtosis quantifies tail weight.
A working rule for reading the numbers:

| \|skew\| | Shape | What to do |
|---|---|---|
| < 0.5 | roughly symmetric | mean and standard deviation are fine |
| 0.5 – 1.0 | moderately skewed | prefer median; consider a mild transform |
| > 1.0 | strongly skewed | use median and IQR; transform before modelling |

In [ ]:
# ---- Skewness, kurtosis, shape classification --------------------------
def classify_shape(s):
    a = abs(s)
    if a < 0.5:  return "roughly symmetric"
    if a < 1.0:  return "moderately skewed"
    return "strongly skewed"

shape = pd.DataFrame({
    "skew"    : df[numeric_cols].skew(),
    "kurtosis": df[numeric_cols].kurtosis(),
    "zeros_%" : (df[numeric_cols] == 0).mean() * 100,
})
shape["direction"] = np.where(shape["skew"] > 0, "right tail", "left tail")
shape["shape"]     = shape["skew"].apply(classify_shape)
shape = shape.sort_values("skew", ascending=False)
print(shape.to_string())

strong = shape[shape["shape"] == "strongly skewed"].index.tolist()
print(f"\n{len(strong)} of {len(numeric_cols)} numeric columns are strongly skewed:")
print(" ", strong)
print("\nCarry this list to P23-24, where these columns are transformed.")

### Histogram, boxplot and density

Three views of the same variable, each hiding something the others reveal:

- **Histogram** — shows gaps and multiple peaks, but the story changes with bin width.
- **Boxplot** — compact five-number summary, ideal for comparing groups; hides multimodality completely.
- **Density (KDE)** — smooth shape, easy to overlay; the smoothness is invented by the bandwidth.

Always look at at least two.

In [ ]:
# ---- Histogram, boxplot, density for one variable ----------------------
col = "ProductRelated_Duration"

fig, axes = plt.subplots(1, 3, figsize=(13, 3.4))
sns.histplot(df[col], bins=50, ax=axes[0], color="#3B6E8F")
axes[0].set_title(f"Histogram — {col}")
sns.boxplot(x=df[col], ax=axes[1], color="#3B6E8F")
axes[1].set_title(f"Boxplot — skew {df[col].skew():.2f}")
sns.kdeplot(df[col], ax=axes[2], fill=True, color="#3B6E8F")
axes[2].set_title("Density (KDE)")
plt.tight_layout(); plt.show()

print("Observe: the histogram is unreadable because a few very long sessions stretch the axis.")
print("The boxplot shows the same fact compactly. The KDE spills below zero, which is impossible")
print("for a duration — an artefact of smoothing, not a property of the data.")

In [ ]:
# ---- The same three views for every strongly skewed column -------------
show = strong[:6] if len(strong) >= 6 else strong
fig, axes = plt.subplots(len(show), 2, figsize=(11, 2.4 * len(show)))
axes = np.atleast_2d(axes)

for i, c in enumerate(show):
    sns.histplot(df[c], bins=40, ax=axes[i, 0], color="#3B6E8F")
    axes[i, 0].set_title(f"{c} — skew {df[c].skew():.2f}", loc="left")
    axes[i, 0].set_ylabel("")
    sns.boxplot(x=df[c], ax=axes[i, 1], color="#8B9199")
    axes[i, 1].set_title("")
    axes[i, 1].set_xlabel("")

plt.tight_layout(); plt.show()

### Pivot tables

A pivot table is non-graphical bivariate EDA: it summarises one variable across the levels of
others. Use it to quantify what a plot suggests, and to produce numbers you can put in a report.

In [ ]:
# ---- Pivot tables ------------------------------------------------------
# 1 — conversion rate by visitor type and weekend
pivot1 = pd.pivot_table(df, values="Revenue", index="VisitorType",
                        columns="Weekend", aggfunc="mean") * 100
print("Conversion rate (%) by visitor type and weekend")
print(pivot1.round(2).to_string(), "\n")

# 2 — median page value and session depth by month
pivot2 = pd.pivot_table(
    df, index="Month",
    values=["PageValues", "ProductRelated", "ProductRelated_Duration"],
    aggfunc="median")
print("Median engagement by month")
print(pivot2.round(2).to_string(), "\n")

# 3 — conversion and volume together: rate alone can mislead
pivot3 = df.groupby("Month").agg(
    sessions=("Revenue", "size"),
    purchases=("Revenue", "sum"),
    conversion_pct=("Revenue", lambda s: 100 * s.mean()),
).sort_values("conversion_pct", ascending=False)
print("Volume and conversion by month")
print(pivot3.round(2).to_string())

print("\nInterpretation")
print("- Always report the denominator. A high conversion rate on 200 sessions is weaker")
print("  evidence than a slightly lower rate on 3,000 sessions.")

### Deliverable — P1-2

One notebook, run top to bottom, containing:

1. The problem statement in your own words, with the sampling frame stated.
2. The dependent / independent split, justified.
3. A table of mean vs median and variance vs IQR for every numeric column.
4. The skewness table with each column classified by shape.
5. Histogram, boxplot and density for at least four variables.
6. At least two pivot tables.
7. **One sentence under every figure and table saying what it shows about the business question.**

---

# P3-4 · Type handling, text and time series

**Week 2 · Module 1 · CO1**

**Objective.** Audit and correct data types, measure word frequency in free text, plot a time series, and quantify categorical imbalance.


### Check datatypes

The dtype pandas guessed is not necessarily the *measurement scale* of the variable.
`OperatingSystems`, `Browser`, `Region` and `TrafficType` are stored as integers but are
**nominal codes** — averaging them is meaningless. This is the single most common error in
submitted assignments.

In [ ]:
# ---- Audit what pandas guessed -----------------------------------------
audit = pd.DataFrame({
    "pandas_dtype": df.dtypes.astype(str),
    "unique"      : df.nunique(),
    "sample"      : [df[c].dropna().unique()[:4] for c in df.columns],
})

# a low unique count on an integer column is a strong hint that it is a code, not a quantity
audit["suspect_nominal"] = (
    df.dtypes.isin([np.dtype("int64")]) & (df.nunique() <= 25)
)
print(audit.to_string())

print("\nColumns stored as integers but almost certainly nominal codes:")
print(" ", audit[audit["suspect_nominal"]].index.tolist())

In [ ]:
# ---- Convert types -----------------------------------------------------
dfc = df.copy()

# 1 — nominal codes: integer -> category (blocks accidental arithmetic)
nominal_codes = ["OperatingSystems", "Browser", "Region", "TrafficType"]
for c in nominal_codes:
    dfc[c] = dfc[c].astype("category")

# 2 — Month is ordinal: give it a real order so it sorts and plots correctly
month_order = ["Jan", "Feb", "Mar", "Apr", "May", "June",
               "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]
present = [m for m in month_order if m in dfc["Month"].unique()]
dfc["Month"] = pd.Categorical(dfc["Month"], categories=present, ordered=True)

# 3 — text categories: strip and standardise before anything counts them
dfc["VisitorType"] = dfc["VisitorType"].astype(str).str.strip()

# 4 — booleans stay boolean
for c in ["Weekend", "Revenue"]:
    dfc[c] = dfc[c].astype(bool)

print("After conversion:")
print(dfc.dtypes.astype(str).to_string())
print("\nMonth is now ordered:", list(dfc["Month"].cat.categories))
print("Months absent from the data:", [m for m in month_order if m not in present])

In [ ]:
# ---- Why the ordered category matters ----------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))

# alphabetical (wrong) vs chronological (right)
df["Month"].value_counts().plot(kind="bar", ax=axes[0], color="#8B9199")
axes[0].set_title("Unordered — pandas sorts by frequency or alphabetically")

dfc["Month"].value_counts().sort_index().plot(kind="bar", ax=axes[1], color="#3B6E8F")
axes[1].set_title("Ordered Categorical — calendar order, trend readable")

plt.tight_layout(); plt.show()
print("The left chart makes seasonality invisible. Same data, different dtype.")

### Text word frequency

The shoppers dataset has no free text, so this section uses the **Amazon consumer reviews**
corpus from the course Drive. If the file is absent the cell falls back to a small built-in
corpus so the notebook still runs.

In [ ]:
# ---- Load a text column ------------------------------------------------
text_path = find("Datafiniti_Amazon_Consumer_Reviews_of_Amazon_Products.csv")

if text_path:
    reviews = pd.read_csv(text_path, usecols=["reviews.text"], low_memory=False)
    corpus  = reviews["reviews.text"].dropna().astype(str)
    print(f"Loaded {len(corpus):,} reviews from {os.path.basename(text_path)}")
else:
    corpus = pd.Series([
        "the website is easy to use and checkout was fast",
        "great discounts on products but delivery was slow",
        "customer service did not respond to my query",
        "excellent product quality and fast delivery",
        "the checkout page kept failing on mobile",
    ])
    print("Amazon file not found — using a 5-document fallback corpus.")

print("\nDocument length (characters):")
print(corpus.str.len().describe().round(1).to_string())

In [ ]:
# ---- Word frequency ----------------------------------------------------
import re
from collections import Counter

STOP = set("""a an the and or but if of to in on for with at by from is are was were be been
being it its this that these those i you he she they we as not no so than then there here
have has had do does did will would can could my your our their me him her them what which
who when where why how all any both each more most other some such only own same too very
s t don now just""".split())

def tokenise(s):
    return [w for w in re.findall(r"[a-z']+", s.lower()) if len(w) > 2 and w not in STOP]

sample  = corpus.sample(min(5000, len(corpus)), random_state=RANDOM_STATE)
counts  = Counter()
for doc in sample:
    counts.update(tokenise(doc))

freq = (pd.Series(counts).sort_values(ascending=False).head(20)
          .rename("count").to_frame())
freq["share_%"] = (freq["count"] / sum(counts.values()) * 100).round(2)
print(f"Vocabulary size: {len(counts):,} distinct tokens")
print(freq.to_string())

plt.figure(figsize=(9, 4))
sns.barplot(x=freq["count"], y=freq.index, color="#3B6E8F")
plt.title("Top 20 tokens (stop words removed)"); plt.xlabel("count"); plt.ylabel("")
plt.tight_layout(); plt.show()

print("\nA ranked frequency bar is precise. A word cloud is decorative — it cannot be read off.")

### Time-series plotting

`HistoricalPrices.csv` holds a daily price series. Time-ordered data has a rule the rest of the
course depends on: **you may not shuffle the rows, and you may not split them randomly.**

In [ ]:
# ---- Time series -------------------------------------------------------
ts_path = find("HistoricalPrices.csv")

if ts_path:
    ts = pd.read_csv(ts_path)
    ts.columns = [c.strip() for c in ts.columns]      # this file has padded headers
    ts["Date"] = pd.to_datetime(ts["Date"], errors="coerce")
    ts = ts.dropna(subset=["Date"]).sort_values("Date").set_index("Date")
    series = ts["Close"]
    label  = "Daily close"
else:
    idx = pd.date_range("2020-01-01", periods=900, freq="D")
    series = pd.Series(np.cumsum(np.random.randn(900)) + 100, index=idx)
    label  = "Synthetic series (HistoricalPrices.csv not found)"

roll30, roll90 = series.rolling(30).mean(), series.rolling(90).mean()

plt.figure(figsize=(11, 3.6))
plt.plot(series.index, series.values, lw=0.8, color="#8B9199", label=label)
plt.plot(roll30.index, roll30.values, lw=1.8, color="#3B6E8F", label="30-day mean")
plt.plot(roll90.index, roll90.values, lw=1.8, color="#B5432E", label="90-day mean")
plt.legend(); plt.title("Level and trend"); plt.tight_layout(); plt.show()

print("Span:", series.index.min().date(), "to", series.index.max().date(),
      f"| {len(series):,} observations")
print("Rolling means separate long-run trend from day-to-day noise without any modelling.")

In [ ]:
# ---- Seasonality in the shoppers data ----------------------------------
monthly = (dfc.groupby("Month", observed=True)
              .agg(sessions=("Revenue", "size"),
                   conversion=("Revenue", "mean")))
monthly["conversion"] *= 100

fig, ax1 = plt.subplots(figsize=(10, 3.4))
ax1.bar(monthly.index.astype(str), monthly["sessions"], color="#C9D4DB", label="sessions")
ax1.set_ylabel("sessions"); ax1.set_xlabel("")
ax2 = ax1.twinx()
ax2.plot(monthly.index.astype(str), monthly["conversion"], color="#B5432E",
         marker="o", lw=2, label="conversion %")
ax2.set_ylabel("conversion %")
ax1.set_title("Volume and conversion by month")
plt.tight_layout(); plt.show()

print(monthly.round(2).to_string())
print("\nNote the dual axis. It is used here deliberately and labelled — but a dual axis can")
print("manufacture an apparent relationship by choosing the two ranges. Use it sparingly.")

### Categorical distribution and imbalance

Imbalance is a univariate finding with consequences that reach all the way to model evaluation.
Record it now.

In [ ]:
# ---- Categorical imbalance --------------------------------------------
cat_cols = ["VisitorType", "Weekend", "Month", "OperatingSystems",
            "Browser", "Region", "TrafficType"]

rows = []
for c in cat_cols:
    vc = dfc[c].value_counts(normalize=True, dropna=False)
    rows.append({
        "column"        : c,
        "levels"        : dfc[c].nunique(),
        "top_level"     : str(vc.index[0]),
        "top_share_%"   : round(vc.iloc[0] * 100, 1),
        "levels_under_1%": int((vc < 0.01).sum()),
    })
imbalance = pd.DataFrame(rows).sort_values("top_share_%", ascending=False)
print(imbalance.to_string(index=False))

print("\nTarget:", f"{y.mean()*100:.1f}% positive")
print("\nInterpretation")
print("- top_share_% above ~90 means the column is nearly constant and carries little signal.")
print("- levels_under_1% counts rare categories: group these into 'Other' before encoding (P21-22).")

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))
for ax, c in zip(axes, ["VisitorType", "Weekend", "Region"]):
    (dfc[c].value_counts(normalize=True) * 100).head(10).plot(
        kind="bar", ax=ax, color="#3B6E8F")
    ax.set_title(f"{c} (% of sessions)"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

### Deliverable — P3-4

A notebook plus a short table listing **every type correction you made and why**, together with
the word-frequency plot, the time-series plot and the imbalance table.

---

# P5-6 · Unstructured data — image and audio

**Week 3 · Module 1 · CO1**

**Objective.** Represent an image as a pixel matrix and an audio file as a waveform, and explore each with the same distributional questions used for tabular data.


### Image pixel distribution

An image is a NumPy array: `(height, width)` for grayscale, `(height, width, 3)` for RGB.
Once it is an array, the same exploratory questions apply — what is the distribution, where are
the extremes, is anything clipped?

In [ ]:
# ---- Load an image -----------------------------------------------------
from PIL import Image

img_path = find("1.png")
if img_path:
    img = np.array(Image.open(img_path).convert("RGB"))
    src = os.path.basename(img_path)
else:
    from skimage import data
    img = data.astronaut()                 # built-in sample, always available
    src = "skimage.data.astronaut()"

print("Source :", src)
print("Shape  :", img.shape, "-> height x width x channels")
print("dtype  :", img.dtype, "| range", img.min(), "to", img.max())
print("Pixels :", f"{img.shape[0] * img.shape[1]:,} per channel")

plt.figure(figsize=(4.2, 4.2))
plt.imshow(img); plt.axis("off"); plt.title(f"Source image — {src}")
plt.tight_layout(); plt.show()

In [ ]:
# ---- Pixel intensity distribution --------------------------------------
gray = np.array(Image.fromarray(img).convert("L"))

fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))

axes[0].imshow(gray, cmap="gray"); axes[0].axis("off")
axes[0].set_title("Grayscale")

axes[1].hist(gray.ravel(), bins=64, color="#3B6E8F")
axes[1].set_title(f"Intensity histogram — mean {gray.mean():.1f}")
axes[1].set_xlabel("pixel value (0-255)")

for ch, colour in zip(range(3), ["#B5432E", "#2E7D4F", "#3B6E8F"]):
    axes[2].hist(img[:, :, ch].ravel(), bins=64, alpha=0.55, color=colour,
                 label=["Red", "Green", "Blue"][ch])
axes[2].legend(); axes[2].set_title("Per-channel distribution")
plt.tight_layout(); plt.show()

stats = pd.DataFrame({
    "channel": ["Red", "Green", "Blue", "Gray"],
    "mean"   : [img[:, :, 0].mean(), img[:, :, 1].mean(), img[:, :, 2].mean(), gray.mean()],
    "std"    : [img[:, :, 0].std(),  img[:, :, 1].std(),  img[:, :, 2].std(),  gray.std()],
    "min"    : [img[:, :, 0].min(),  img[:, :, 1].min(),  img[:, :, 2].min(),  gray.min()],
    "max"    : [img[:, :, 0].max(),  img[:, :, 1].max(),  img[:, :, 2].max(),  gray.max()],
})
print(stats.round(1).to_string(index=False))

clipped_low  = (gray == 0).mean() * 100
clipped_high = (gray == 255).mean() * 100
print(f"\nClipped at 0: {clipped_low:.2f}% of pixels | clipped at 255: {clipped_high:.2f}%")
print("Heavy clipping means detail was lost at capture and cannot be recovered.")
print("A large gap between channel means indicates a colour cast.")

### Simple audio waveform plotting

Audio is amplitude sampled over time: a 1-D array plus a **sample rate**. The standard library
`wave` module reads uncompressed WAV files, so no extra install is needed.

In [ ]:
# ---- Load audio and plot the waveform ----------------------------------
import wave

wav_path = find("1.wav")

if wav_path:
    with wave.open(wav_path, "rb") as w:
        n_channels, sampwidth = w.getnchannels(), w.getsampwidth()
        rate, n_frames        = w.getframerate(), w.getnframes()
        raw = w.readframes(n_frames)
    dtype  = {1: np.uint8, 2: np.int16, 4: np.int32}[sampwidth]
    signal = np.frombuffer(raw, dtype=dtype).astype(np.float32)
    if n_channels > 1:                       # average the channels down to mono
        signal = signal.reshape(-1, n_channels).mean(axis=1)
    signal = signal / (np.abs(signal).max() or 1)     # normalise to -1..1
    src = os.path.basename(wav_path)
else:
    rate, n_channels = 22050, 1
    t = np.linspace(0, 4, rate * 4, endpoint=False)
    signal = (0.6 * np.sin(2 * np.pi * 220 * t) *
              np.exp(-0.4 * t) + 0.03 * np.random.randn(t.size))
    src = "synthetic 220 Hz tone (1.wav not found)"

duration = len(signal) / rate
time     = np.arange(len(signal)) / rate

print("Source      :", src)
print("Sample rate :", f"{rate:,} Hz")
print("Channels    :", n_channels, "(mixed to mono for analysis)")
print("Duration    :", f"{duration:.2f} s")
print("Samples     :", f"{len(signal):,}")

In [ ]:
# ---- Waveform and amplitude distribution -------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 3.2))

axes[0].plot(time, signal, lw=0.4, color="#3B6E8F")
axes[0].set_title("Waveform"); axes[0].set_xlabel("seconds"); axes[0].set_ylabel("amplitude")

axes[1].hist(signal, bins=80, color="#3B6E8F")
axes[1].set_title("Amplitude distribution"); axes[1].set_xlabel("amplitude")

# short-term energy: loudness over time, in 50 ms frames
frame = int(0.05 * rate)
n_win = len(signal) // frame
energy = np.array([np.sqrt(np.mean(signal[i*frame:(i+1)*frame] ** 2))
                   for i in range(n_win)])
axes[2].plot(np.arange(n_win) * 0.05, energy, color="#B5432E")
axes[2].set_title("Short-term energy (RMS, 50 ms frames)"); axes[2].set_xlabel("seconds")

plt.tight_layout(); plt.show()

silence = (energy < 0.02).mean() * 100
print(f"Peak amplitude   : {np.abs(signal).max():.3f}")
print(f"RMS level        : {np.sqrt(np.mean(signal ** 2)):.3f}")
print(f"Near-silent time : {silence:.1f}% of frames below 0.02 RMS")
print("\nThe amplitude histogram is centred on zero and symmetric — as sound must be.")
print("Energy over time shows structure the raw waveform is too dense to reveal.")

### Deliverable — P5-6

One notebook covering both modalities, stating **shape, dtype and value range explicitly** for
each, plus a short note on what exploratory question each representation makes answerable.

---

# P7-8 · Univariate EDA

**Week 4 · Module 2 · CO2**

**Objective.** Run a complete univariate sweep, compare the available distribution plots on the same variable, and interpret the findings.


### Perform univariate EDA

A univariate sweep answers four questions for every column: **what is the centre, what is the
spread, what is the shape, and what is unusual?** Write it once as a function and reuse it in
every project including the final one.

In [ ]:
# ---- A reusable univariate sweep ---------------------------------------
def univariate_sweep(frame, cols=None):
    """Centre, spread, shape and anomaly flags for every numeric column."""
    cols = cols or frame.select_dtypes(include=[np.number]).columns.tolist()
    out = []
    for c in cols:
        s   = frame[c].dropna()
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        out.append({
            "column"    : c,
            "n"         : len(s),
            "mean"      : s.mean(),
            "median"    : s.median(),
            "std"       : s.std(),
            "IQR"       : iqr,
            "skew"      : s.skew(),
            "kurtosis"  : s.kurtosis(),
            "zeros_%"   : (s == 0).mean() * 100,
            "iqr_out_%" : (((s < q1 - 1.5*iqr) | (s > q3 + 1.5*iqr)).mean() * 100),
        })
    return pd.DataFrame(out).set_index("column")

sweep = univariate_sweep(df, numeric_cols)
print(sweep.round(2).to_string())

In [ ]:
# ---- Flag what needs attention ------------------------------------------
flags = pd.DataFrame({
    "strong_skew"   : sweep["skew"].abs() > 1,
    "heavy_tails"   : sweep["kurtosis"] > 3,
    "zero_inflated" : sweep["zeros_%"] > 50,
    "many_outliers" : sweep["iqr_out_%"] > 5,
})
flags["issues"] = flags.sum(axis=1)
print(flags.sort_values("issues", ascending=False).to_string())

print("\nColumns needing treatment before modelling:")
print(" ", flags[flags["issues"] >= 2].index.tolist())

### Compare different distribution plots

Five ways to show one variable. Each is honest; each hides something.

In [ ]:
# ---- Five views of one variable ----------------------------------------
col = "BounceRates"
s   = df[col]

fig, axes = plt.subplots(1, 5, figsize=(15, 2.9))

sns.histplot(s, bins=40, ax=axes[0], color="#1F6F6B")
axes[0].set_title("Histogram (40 bins)")

sns.histplot(s, bins=8, ax=axes[1], color="#1F6F6B")
axes[1].set_title("Histogram (8 bins)")

sns.kdeplot(s, ax=axes[2], fill=True, color="#1F6F6B")
axes[2].set_title("Density (KDE)")

sns.boxplot(x=s, ax=axes[3], color="#1F6F6B")
axes[3].set_title("Boxplot")

sns.ecdfplot(s, ax=axes[4], color="#1F6F6B")
axes[4].set_title("ECDF")

for a in axes: a.set_xlabel(col); a.set_ylabel("")
plt.tight_layout(); plt.show()

print("Same variable, five renderings:")
print("- 40 bins shows the spike at zero; 8 bins conceals it entirely.")
print("- The KDE invents density below zero, which is impossible for a rate.")
print("- The boxplot compresses everything into five numbers and a cloud of flagged points.")
print("- The ECDF needs no binning choice and reads percentiles exactly.")

In [ ]:
# ---- Univariate comparison across the target groups --------------------
show = ["PageValues", "ProductRelated", "BounceRates", "ExitRates"]
fig, axes = plt.subplots(2, len(show), figsize=(14, 5.6))

for j, c in enumerate(show):
    sns.kdeplot(data=df, x=c, hue="Revenue", ax=axes[0, j],
                fill=True, alpha=0.35, common_norm=False)
    axes[0, j].set_title(c); axes[0, j].set_ylabel("")
    sns.boxplot(data=df, x="Revenue", y=c, ax=axes[1, j], showfliers=False)
    axes[1, j].set_title("")

plt.tight_layout(); plt.show()

comparison = df.groupby("Revenue")[show].median().T
comparison.columns = ["no purchase", "purchase"]
comparison["ratio"] = comparison["purchase"] / comparison["no purchase"].replace(0, np.nan)
print(comparison.round(3).to_string())

### Interpret findings

A figure without a written interpretation earns no marks. Convert each observation into a claim
about the business question.

In [ ]:
# ---- Written interpretation --------------------------------------------
findings = [
    ("PageValues",
     "Median is 0 for the large majority of sessions but clearly positive for converting "
     "sessions. This is the strongest single separator in the dataset."),
    ("ProductRelated / _Duration",
     "Both are strongly right-skewed and zero-inflated. Converting sessions view more product "
     "pages and stay longer, but the tail is extreme — transform before modelling (P23-24)."),
    ("BounceRates / ExitRates",
     "Both spike at zero and are bounded in [0,1]. Converting sessions have visibly lower "
     "values. The two are near-duplicates of each other — check in P9-10."),
    ("SpecialDay",
     "Almost entirely zero. Nearly constant, so it carries little information on its own."),
    ("Administrative / Informational",
     "Low counts, heavily zero-inflated. Candidates for combining into a single "
     "'non-product page depth' feature in P25-26."),
]
for name, text in findings:
    print(f"- {name}\n    {textwrap.fill(text, 92, subsequent_indent='    ')}")

### Deliverable — P7-8

A notebook plus a **one-page findings table, one row per variable**, giving centre, spread, shape,
anomalies and a one-line interpretation.

---

# P9-10 · Bivariate analysis and hypothesis generation

**Week 5 · Module 2 · CO2**

**Objective.** Build and read a correlation matrix, use heatmaps and cross-tabulation, and convert observed relationships into testable hypotheses.


### Correlation matrix

Correlation is standardised covariance, bounded in [-1, 1] and unitless. Pearson measures
**linear** association only; Spearman works on ranks and catches any monotonic relationship.
Comparing the two is a cheap non-linearity test.

In [ ]:
# ---- Correlation matrix, two ways --------------------------------------
num = df[numeric_cols + [TARGET]].copy()
num[TARGET] = num[TARGET].astype(int)

pearson  = num.corr(method="pearson")
spearman = num.corr(method="spearman")

target_corr = pd.DataFrame({
    "pearson" : pearson[TARGET].drop(TARGET),
    "spearman": spearman[TARGET].drop(TARGET),
})
target_corr["gap"] = (target_corr["spearman"] - target_corr["pearson"]).abs()
target_corr = target_corr.reindex(target_corr["spearman"].abs()
                                  .sort_values(ascending=False).index)
print("Association with the target")
print(target_corr.round(3).to_string())

print("\nInterpretation")
print("- A large pearson/spearman gap means the relationship is monotonic but not linear.")
print("- PageValues shows the largest gap: strong in rank terms, weaker as a straight line.")

### Heatmaps

In [ ]:
# ---- Heatmap of the full matrix ----------------------------------------
mask = np.triu(np.ones_like(pearson, dtype=bool))

fig, axes = plt.subplots(1, 2, figsize=(15, 5.4))
sns.heatmap(pearson, mask=mask, annot=True, fmt=".2f", center=0, cmap="RdBu_r",
            vmin=-1, vmax=1, square=False, ax=axes[0], cbar_kws={"shrink": .7},
            annot_kws={"size": 7})
axes[0].set_title("Pearson correlation")

sns.heatmap(spearman, mask=mask, annot=True, fmt=".2f", center=0, cmap="RdBu_r",
            vmin=-1, vmax=1, square=False, ax=axes[1], cbar_kws={"shrink": .7},
            annot_kws={"size": 7})
axes[1].set_title("Spearman correlation")
plt.tight_layout(); plt.show()

In [ ]:
# ---- Multicollinearity among the predictors ----------------------------
pred = pearson.drop(index=TARGET, columns=TARGET)
pairs = (pred.where(np.triu(np.ones(pred.shape), k=1).astype(bool))
             .stack().rename("r").reset_index())
pairs.columns = ["var_1", "var_2", "r"]
pairs["abs_r"] = pairs["r"].abs()
print("Most correlated predictor pairs")
print(pairs.sort_values("abs_r", ascending=False).head(8).round(3).to_string(index=False))

# VIF from the inverse correlation matrix — no statsmodels needed
def vif_table(frame):
    c = frame.corr().values
    inv = np.linalg.pinv(c)
    return (pd.Series(np.diag(inv), index=frame.columns, name="VIF")
              .sort_values(ascending=False).to_frame())

vif = vif_table(df[numeric_cols])
vif["verdict"] = pd.cut(vif["VIF"], [0, 5, 10, np.inf],
                        labels=["ok", "investigate", "severe"])
print("\nVariance inflation factor")
print(vif.round(2).to_string())
print("\nVIF > 10 means the column is largely reconstructable from the others.")
print("ExitRates and BounceRates measure nearly the same thing — keep one, or combine them.")

### Cross-tab analysis

For two categorical variables, counts mislead and **percentages inform**. Always normalise along
the axis that answers your question, and always report the group sizes.

In [ ]:
# ---- Cross-tabulation ---------------------------------------------------
def crosstab_report(frame, cat, target=TARGET, min_n=30):
    counts = pd.crosstab(frame[cat], frame[target])
    rate   = pd.crosstab(frame[cat], frame[target], normalize="index") * 100
    out = pd.DataFrame({
        "sessions"      : counts.sum(axis=1),
        "purchases"     : counts.get(True, 0),
        "conversion_%"  : rate.get(True, 0).round(2),
    })
    out["reliable"] = out["sessions"] >= min_n
    return out.sort_values("conversion_%", ascending=False)

for c in ["VisitorType", "Weekend", "Month"]:
    print(f"--- {c} vs {TARGET}")
    print(crosstab_report(dfc, c).to_string(), "\n")

In [ ]:
# ---- Chi-square and effect size ----------------------------------------
from scipy.stats import chi2_contingency

def cramers_v(frame, a, b):
    tab = pd.crosstab(frame[a], frame[b])
    chi2, p, dof, _ = chi2_contingency(tab)
    n = tab.values.sum()
    v = np.sqrt(chi2 / (n * (min(tab.shape) - 1)))
    return chi2, p, v

rows = []
for c in ["VisitorType", "Weekend", "Month", "Region", "TrafficType", "Browser"]:
    chi2, p, v = cramers_v(dfc, c, TARGET)
    rows.append({"variable": c, "chi2": chi2, "p_value": p, "cramers_v": v,
                 "strength": "negligible" if v < .1 else
                             "weak" if v < .2 else
                             "moderate" if v < .3 else "strong"})
assoc = pd.DataFrame(rows).sort_values("cramers_v", ascending=False)
print(assoc.round(4).to_string(index=False))

print("\nInterpretation")
print("- Chi-square says whether an association exists; Cramer's V says how strong it is.")
print("- With 12,330 rows almost everything is 'significant'. Read the effect size, not the p-value.")

In [ ]:
# ---- Visualise the strongest relationships -----------------------------
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))

ct = pd.crosstab(dfc["VisitorType"], dfc[TARGET], normalize="index") * 100
ct.plot(kind="bar", stacked=True, ax=axes[0],
        color=["#C9D4DB", "#1F6F6B"], legend=False)
axes[0].set_title("Conversion by visitor type (%)"); axes[0].set_xlabel("")

sns.boxplot(data=df, x=TARGET, y="PageValues", ax=axes[1], showfliers=False)
axes[1].set_title("PageValues by outcome")

sns.scatterplot(data=df.sample(3000, random_state=RANDOM_STATE),
                x="BounceRates", y="ExitRates", hue=TARGET,
                alpha=0.4, s=12, ax=axes[2])
axes[2].set_title("BounceRates vs ExitRates (r = %.2f)"
                  % df["BounceRates"].corr(df["ExitRates"]))
plt.tight_layout(); plt.show()

### Hypothesis generation

A hypothesis names a **mechanism**, predicts a **direction**, and states what would **falsify** it.
"X is related to Y" is a restatement of the correlation, not a hypothesis.

In [ ]:
# ---- Hypotheses from the evidence above --------------------------------
hypotheses = [
    dict(id="H1",
         claim="Sessions that reach a page carrying a positive PageValue are far more likely to "
               "convert, because PageValues is assigned to pages on the checkout path.",
         direction="PageValues > 0 raises conversion",
         evidence="Spearman with target is the largest of any variable; medians differ sharply",
         falsified_by="Converting and non-converting sessions show the same PageValues "
                      "distribution once session depth is controlled for",
         confounder="Session depth — deeper sessions reach more pages of every kind"),
    dict(id="H2",
         claim="Returning visitors convert at a higher rate than new visitors because they have "
               "already evaluated the retailer on an earlier visit.",
         direction="Returning_Visitor > New_Visitor",
         evidence="Cross-tab conversion rates differ by visitor type; Cramer's V is non-negligible",
         falsified_by="The difference disappears after conditioning on month and traffic type",
         confounder="Acquisition channel: new visitors may arrive from lower-intent campaigns"),
    dict(id="H3",
         claim="Conversion is seasonal, peaking in the November-December shopping period.",
         direction="Nov and Dec above the annual mean",
         evidence="Monthly conversion series is not flat; volume also rises",
         falsified_by="Monthly rates fall within sampling variation of the overall rate",
         confounder="Promotional calendar is not observed in this dataset"),
]

for h in hypotheses:
    print(f"[{h['id']}] {textwrap.fill(h['claim'], 92, subsequent_indent='     ')}")
    print(f"     direction    : {h['direction']}")
    print(f"     evidence     : {h['evidence']}")
    print(f"     falsified by : {textwrap.fill(h['falsified_by'], 74, subsequent_indent=' '*20)}")
    print(f"     confounder   : {h['confounder']}\n")

### Deliverable — P9-10

A notebook ending in a **numbered list of at least three hypotheses**, each with its direction,
supporting evidence, falsification condition and named confounders.

---

# P11-12 · Grouping — K-means, silhouette and pair plots

**Week 6 · Module 2 · CO2**

**Objective.** Use clustering exploratorily to discover structure, evaluate whether that structure is real, and profile and visualise the groups found.


### Implement K-means for structure discovery

K-means minimises Euclidean distance, so **features must be scaled first** or the column with the
largest range decides the clusters by itself. In this course clustering is exploratory: a way of
seeing structure, not a predictive model.

In [ ]:
# ---- Prepare and scale --------------------------------------------------
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

cluster_cols = ["Administrative", "Administrative_Duration", "Informational",
                "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
                "BounceRates", "ExitRates", "PageValues"]

Xc = df[cluster_cols].copy()
print("Raw ranges — why scaling is not optional:")
print(Xc.agg(["min", "max", "std"]).T.round(2).to_string())

scaler = StandardScaler()
Xs = scaler.fit_transform(Xc)
print("\nAfter standardising: mean ~0, std ~1 for every column.")

In [ ]:
# ---- Choose k: inertia and silhouette ----------------------------------
ks, inertias, sils = range(2, 9), [], []

for k in ks:
    km  = KMeans(n_clusters=k, n_init=10, random_state=RANDOM_STATE)
    lab = km.fit_predict(Xs)
    inertias.append(km.inertia_)
    # silhouette on a sample keeps this fast on 12k rows
    sils.append(silhouette_score(Xs, lab, sample_size=4000, random_state=RANDOM_STATE))

choice = pd.DataFrame({"k": list(ks), "inertia": inertias, "silhouette": sils})
print(choice.round(4).to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.2))
axes[0].plot(list(ks), inertias, marker="o", color="#1F6F6B")
axes[0].set_title("Elbow — within-cluster sum of squares"); axes[0].set_xlabel("k")
axes[1].plot(list(ks), sils, marker="o", color="#B5432E")
axes[1].set_title("Mean silhouette"); axes[1].set_xlabel("k")
plt.tight_layout(); plt.show()

best_k = int(choice.loc[choice["silhouette"].idxmax(), "k"])
print(f"Highest silhouette at k = {best_k} ({max(sils):.3f})")

### Silhouette score (basic)

Silhouette compares how close a point is to its own cluster versus the nearest other cluster.
It runs from -1 to +1:

| Score | Reading |
|---|---|
| > 0.7 | strong, well-separated clusters |
| 0.5 – 0.7 | reasonable structure |
| 0.25 – 0.5 | weak; the clusters overlap |
| < 0.25 | no substantial structure — say so |

**Reporting a weak score honestly is the correct answer.** Manufacturing clusters that are not
there is not.

In [ ]:
# ---- Fit the chosen solution and inspect the silhouette ----------------
from sklearn.metrics import silhouette_samples

km     = KMeans(n_clusters=best_k, n_init=10, random_state=RANDOM_STATE)
labels = km.fit_predict(Xs)
df_cl  = df.copy()
df_cl["cluster"] = labels

samp_idx = np.random.RandomState(RANDOM_STATE).choice(len(Xs), 4000, replace=False)
sv = silhouette_samples(Xs[samp_idx], labels[samp_idx])
overall = sv.mean()

print(f"Overall mean silhouette: {overall:.3f}")
per_cluster = pd.DataFrame({"cluster": labels[samp_idx], "silhouette": sv}) \
                .groupby("cluster")["silhouette"].agg(["mean", "size"])
print(per_cluster.round(3).to_string())

verdict = ("strong" if overall > .7 else "reasonable" if overall > .5
           else "weak — clusters overlap" if overall > .25 else "no substantial structure")
print(f"\nVerdict: {verdict}.")
print("Report this honestly in your write-up rather than presenting the clusters as clean segments.")

In [ ]:
# ---- Profile the clusters ----------------------------------------------
profile = df_cl.groupby("cluster")[cluster_cols].median()
profile["sessions"]      = df_cl["cluster"].value_counts().sort_index()
profile["share_%"]       = (profile["sessions"] / len(df_cl) * 100).round(1)
profile["conversion_%"]  = (df_cl.groupby("cluster")[TARGET].mean() * 100).round(2)
print("Cluster profile (medians)")
print(profile.T.round(2).to_string())

overall_rate = df_cl[TARGET].mean() * 100
print(f"\nOverall conversion rate: {overall_rate:.2f}%")
print("\nName each cluster from its profile. A cluster you cannot describe in one sentence")
print("should not appear in your report.")

In [ ]:
# ---- Stability check: does the structure survive a different seed? -----
stability = []
for seed in [0, 7, 42, 123]:
    lab = KMeans(n_clusters=best_k, n_init=10, random_state=seed).fit_predict(Xs)
    stability.append({
        "seed": seed,
        "silhouette": silhouette_score(Xs, lab, sample_size=4000, random_state=0),
        "largest_cluster_%": round(pd.Series(lab).value_counts(normalize=True).max()*100, 1),
    })
print(pd.DataFrame(stability).round(3).to_string(index=False))
print("\nStructure that changes materially with the seed was never there.")

### Visualize groups

Nine dimensions cannot be plotted directly, so project onto the first two principal components.
PCA is covered properly in P27-28; here it is used only as a viewing device.

In [ ]:
# ---- 2-D projection coloured by cluster and by outcome -----------------
from sklearn.decomposition import PCA

proj = PCA(n_components=2, random_state=RANDOM_STATE).fit(Xs)
P2   = proj.transform(Xs)
samp = np.random.RandomState(RANDOM_STATE).choice(len(P2), 4000, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
sns.scatterplot(x=P2[samp, 0], y=P2[samp, 1], hue=labels[samp],
                palette="deep", s=10, alpha=0.6, ax=axes[0], legend="full")
axes[0].set_title(f"K-means clusters (k={best_k}) on PC1/PC2")

sns.scatterplot(x=P2[samp, 0], y=P2[samp, 1], hue=df[TARGET].values[samp],
                palette=["#C9D4DB", "#B5432E"], s=10, alpha=0.6, ax=axes[1])
axes[1].set_title("Same projection, coloured by actual outcome")

for a in axes:
    a.set_xlabel(f"PC1 ({proj.explained_variance_ratio_[0]*100:.1f}% var)")
    a.set_ylabel(f"PC2 ({proj.explained_variance_ratio_[1]*100:.1f}% var)")
plt.tight_layout(); plt.show()

print(f"PC1 + PC2 capture {proj.explained_variance_ratio_[:2].sum()*100:.1f}% of the variance.")
print("Compare the two panels: do the discovered clusters align with the outcome, or not?")

### Pair plots

A pair plot is a matrix of scatterplots for every variable pair, with distributions on the
diagonal. It is the densest single view of multivariate structure — and slow, so **always sample**.

In [ ]:
# ---- Pair plot ----------------------------------------------------------
pair_cols = ["ProductRelated", "ProductRelated_Duration", "BounceRates",
             "ExitRates", "PageValues"]

pair_df = df_cl.sample(1500, random_state=RANDOM_STATE)[pair_cols + ["cluster"]]
pair_df["cluster"] = pair_df["cluster"].astype(str)

g = sns.pairplot(pair_df, hue="cluster", corner=True, diag_kind="kde",
                 plot_kws=dict(s=8, alpha=0.35), height=1.5)
g.figure.suptitle("Pair plot by cluster (1,500-row sample)", y=1.01)
plt.show()

print("Read the off-diagonal panels for separation between colours and the diagonal for shape.")
print("Heavy skew makes most panels crowd into one corner — a strong argument for P23-24.")

### Deliverable — P11-12

A notebook containing the k selection evidence, a **named profile for every cluster**, the
stability check across seeds, the 2-D projection, the pair plot, and an **honest verdict on
whether the structure is real**.

---

# P15-16 · Data quality audit

**Week 8 · Module 3 · CO3**

**Objective.** Audit the dataset against the six quality dimensions, remove duplicates, standardise inconsistent encodings, and produce a cleaning log.


### Data quality audit

Six dimensions, audited in this order — validity failures often explain completeness failures,
which in turn explain accuracy failures.

| Dimension | Question |
|---|---|
| Accuracy | Do the values reflect reality? |
| Completeness | Is anything absent that should be present? |
| Consistency | Do values agree across records and sources? |
| Validity | Do values conform to type, range and set rules? |
| Uniqueness | Is each real-world entity represented once? |
| Timeliness | Is the data current, and from one period? |

In [ ]:
# ---- The profile table --------------------------------------------------
def quality_profile(frame):
    prof = pd.DataFrame({
        "dtype"     : frame.dtypes.astype(str),
        "non_null"  : frame.notna().sum(),
        "nulls"     : frame.isna().sum(),
        "null_%"    : (frame.isna().mean() * 100).round(2),
        "unique"    : frame.nunique(),
        "unique_%"  : (frame.nunique() / len(frame) * 100).round(2),
    })
    num = frame.select_dtypes(include=[np.number])
    prof["zeros_%"] = (num == 0).mean().mul(100).round(2)
    prof["min"]     = num.min()
    prof["max"]     = num.max()
    prof["constant"] = frame.nunique() <= 1
    return prof

profile = quality_profile(df)
print(profile.to_string())

print("\nCompleteness: total nulls =", int(df.isna().sum().sum()))
print("Note that zero nulls does NOT mean zero missing values — see P17-18.")

In [ ]:
# ---- Validity rules from domain knowledge -------------------------------
# Write the rules BEFORE looking at the data, then measure the violations.
rules = {
    "BounceRates"            : (0.0, 1.0),      # it is a rate
    "ExitRates"              : (0.0, 1.0),      # it is a rate
    "SpecialDay"             : (0.0, 1.0),      # closeness indicator
    "PageValues"             : (0.0, np.inf),   # cannot be negative
    "Administrative"         : (0, np.inf),     # a count
    "Informational"          : (0, np.inf),
    "ProductRelated"         : (1, np.inf),     # a session must view >= 1 page
    "Administrative_Duration": (0.0, np.inf),   # seconds
    "Informational_Duration" : (0.0, np.inf),
    "ProductRelated_Duration": (0.0, np.inf),
}

violations = []
for col, (lo, hi) in rules.items():
    bad = ~df[col].between(lo, hi)
    violations.append({"column": col, "rule": f"[{lo}, {hi}]",
                       "violations": int(bad.sum()),
                       "pct": round(bad.mean() * 100, 3)})
viol = pd.DataFrame(violations)
print(viol.to_string(index=False))

# cross-field rule: a duration cannot be positive when the page count is zero
cross = pd.DataFrame({
    "rule": ["Administrative_Duration > 0 while Administrative == 0",
             "Informational_Duration > 0 while Informational == 0",
             "ProductRelated_Duration > 0 while ProductRelated == 0"],
    "violations": [
        int(((df["Administrative"] == 0) & (df["Administrative_Duration"] > 0)).sum()),
        int(((df["Informational"] == 0) & (df["Informational_Duration"] > 0)).sum()),
        int(((df["ProductRelated"] == 0) & (df["ProductRelated_Duration"] > 0)).sum()),
    ]})
print("\nCross-field rules")
print(cross.to_string(index=False))

print("\nNegative durations, if any, are sentinel values or clock errors — never real.")

### Duplicate removal

Two kinds matter: **exact duplicates** across all columns, and **key duplicates** where the same
entity appears with differing fields. This dataset has no session id, so an exact duplicate is
either a genuine repeat of identical behaviour or a logging fault — a judgement call you must
record either way.

In [ ]:
# ---- Duplicates ---------------------------------------------------------
exact = df.duplicated()
print(f"Exact duplicate rows: {exact.sum():,} ({exact.mean()*100:.2f}%)")

dup_rows = df[df.duplicated(keep=False)].sort_values(list(df.columns))
print("\nA sample of duplicated rows:")
print(dup_rows.head(6).to_string())

print("\nHow duplicates distribute across the target:")
print(df[df.duplicated(keep=False)][TARGET].value_counts(normalize=True).round(3).to_string())

df_clean = df.drop_duplicates().reset_index(drop=True)
print(f"\nBefore: {len(df):,} rows   After: {len(df_clean):,} rows   "
      f"Removed: {len(df) - len(df_clean):,}")

print("\nEffect on the headline statistic:")
print(f"  conversion rate before {df[TARGET].mean()*100:.3f}%  "
      f"after {df_clean[TARGET].mean()*100:.3f}%")
print("A negligible change means duplicates were not driving the result — record that finding.")

### Standardization

Standardising here means making the *encodings* consistent — case, whitespace, category labels,
units. (Standardising the *scale* of numeric variables is a different operation, covered in P23-24.)

In [ ]:
# ---- Standardise categorical encodings ---------------------------------
def standardise_categories(frame, cols):
    """Trim whitespace, collapse internal spaces, and unify case for label columns."""
    out, log = frame.copy(), []
    for c in cols:
        before = out[c].nunique(dropna=False)
        s = out[c].astype(str).str.strip().str.replace(r"\s+", " ", regex=True)
        out[c] = s
        after = out[c].nunique(dropna=False)
        log.append({"column": c, "levels_before": before, "levels_after": after,
                    "collapsed": before - after})
    return out, pd.DataFrame(log)

df_clean, std_log = standardise_categories(df_clean, ["VisitorType", "Month"])
print(std_log.to_string(index=False))

print("\nValue set after standardisation:")
for c in ["VisitorType", "Month"]:
    print(f"  {c}: {sorted(df_clean[c].unique().tolist())}")

# demonstrate why it matters, on a deliberately dirtied copy
dirty = df[["VisitorType"]].head(200).copy()
dirty.loc[:60,  "VisitorType"] = " Returning_Visitor"
dirty.loc[61:120,"VisitorType"] = "returning_visitor "
print(f"\nDirtied sample: {dirty['VisitorType'].nunique()} apparent levels")
fixed = dirty["VisitorType"].str.strip().str.lower()
print(f"After trim + case-fold: {fixed.nunique()} real levels")
print("Every frequency table computed before this step would have been wrong.")

In [ ]:
# ---- Rare-level grouping: prepare high-cardinality columns -------------
def group_rare(series, threshold=0.01, other="Other"):
    share = series.value_counts(normalize=True)
    rare  = share[share < threshold].index
    return series.where(~series.isin(rare), other), list(rare)

for c in ["TrafficType", "Browser", "OperatingSystems", "Region"]:
    grouped, rare = group_rare(df_clean[c].astype(str))
    print(f"{c:18} {df_clean[c].nunique():3} levels -> {grouped.nunique():3} "
          f"({len(rare)} rare levels grouped into 'Other')")
    df_clean[c + "_grouped"] = grouped

print("\nThreshold used: levels below 1% of rows. Record the threshold — it is a parameter.")

In [ ]:
# ---- The cleaning log ---------------------------------------------------
cleaning_log = pd.DataFrame([
    dict(step=1, dimension="Uniqueness", finding="Exact duplicate rows",
         rows=int(exact.sum()), action="Dropped",
         justification="Identical across all 18 columns; no session id to distinguish them"),
    dict(step=2, dimension="Consistency", finding="Whitespace/case in label columns",
         rows=0, action="Trimmed and collapsed whitespace",
         justification="Prevents the same category being counted as two"),
    dict(step=3, dimension="Validity", finding="Rate columns checked against [0,1]",
         rows=int(viol.loc[viol.column.isin(['BounceRates','ExitRates']), 'violations'].sum()),
         action="Verified, none out of range",
         justification="Domain rule: a rate cannot exceed 1"),
    dict(step=4, dimension="Consistency", finding="Rare categorical levels",
         rows=0, action="Grouped below 1% into 'Other'",
         justification="Avoids column explosion and unstable estimates at encoding"),
    dict(step=5, dimension="Completeness", finding="No explicit nulls",
         rows=0, action="Flagged for P17-18",
         justification="Zero nulls does not prove zero missingness"),
])
print(cleaning_log.to_string(index=False))

print(f"\nFinal cleaned shape: {df_clean.shape}")
print("The cleaning log is a deliverable. It carries most of the marks in Assignment 2.")

### Assignment 1 review

Use this session to review Assignment 1 against the marking scheme:

| Criterion | Weight | What is being judged |
|---|---|---|
| Correctness of technique | 30% | Right method for the variable type; no Pearson on nominal data |
| Interpretation | 35% | Every figure carries a sentence saying what it means |
| Reproducibility | 15% | Runs top to bottom; relative paths; commented; seeded |
| Presentation | 20% | Clear narrative, honest about limits |

**The most common failures:** plots with no written interpretation; averaging ordinal or nominal
codes; absolute paths that only work on the author's machine; and reporting accuracy on an
imbalanced target.

### Deliverable — P15-16

A notebook plus the **cleaning log as a table** — finding, rows affected, action, justification —
with before/after counts for every step.

---

# P17-18 · Missing values and imputation

**Week 9 · Module 3 · CO3**

**Objective.** Compare imputation strategies against a known ground truth and measure what each one does to the variance and shape of the data.


### A controlled experiment

The shoppers dataset has no missing values, which is an opportunity rather than a problem: we can
**delete values ourselves and keep the truth**. That lets us measure imputation error directly,
which is impossible on real missing data.

We inject two mechanisms:

- **MCAR** — values removed completely at random.
- **MAR** — values removed with a probability that depends on *another observed* column.

Compare with the real-world case in `pima.csv`, where zeros in `Glucose`, `BloodPressure`,
`SkinThickness`, `Insulin` and `BMI` are disguised missing values that `isna()` cannot see.

In [ ]:
# ---- Inject missingness with a known ground truth ----------------------
rng = np.random.RandomState(RANDOM_STATE)

work  = df_clean.copy()
truth = work[["ProductRelated_Duration", "BounceRates", "PageValues"]].copy()

# 1 — MCAR: 15% of ProductRelated_Duration removed at random
mcar_mask = rng.rand(len(work)) < 0.15
work.loc[mcar_mask, "ProductRelated_Duration"] = np.nan

# 2 — MAR: BounceRates missing more often for short sessions
p_missing = np.where(work["ProductRelated"] <= 3, 0.35, 0.05)
mar_mask  = rng.rand(len(work)) < p_missing
work.loc[mar_mask, "BounceRates"] = np.nan

# 3 — MNAR-flavoured: high PageValues hidden more often
p_hide     = np.where(truth["PageValues"] > truth["PageValues"].quantile(0.9), 0.5, 0.03)
mnar_mask  = rng.rand(len(work)) < p_hide
work.loc[mnar_mask, "PageValues"] = np.nan

summary = pd.DataFrame({
    "column"    : ["ProductRelated_Duration", "BounceRates", "PageValues"],
    "mechanism" : ["MCAR", "MAR", "MNAR"],
    "missing"   : [mcar_mask.sum(), mar_mask.sum(), mnar_mask.sum()],
    "missing_%" : [mcar_mask.mean()*100, mar_mask.mean()*100, mnar_mask.mean()*100],
})
print(summary.round(2).to_string(index=False))
print("\nTotal nulls now:", int(work.isna().sum().sum()))

In [ ]:
# ---- Detect and diagnose the mechanism ---------------------------------
# Build an indicator per affected column, then explore it like any other variable.
for c in ["ProductRelated_Duration", "BounceRates", "PageValues"]:
    work[c + "_missing"] = work[c].isna().astype(int)

print("Correlation of each missingness indicator with the OBSERVED columns")
probe = ["ProductRelated", "Administrative", "ExitRates", "SpecialDay"]
diag = pd.DataFrame({
    c: work[probe].corrwith(work[c + "_missing"])
    for c in ["ProductRelated_Duration", "BounceRates", "PageValues"]
})
print(diag.round(3).to_string())

print("\nReading the table")
print("- ProductRelated_Duration: no meaningful correlation -> consistent with MCAR.")
print("- BounceRates: correlates with ProductRelated -> MAR, and ProductRelated is the driver.")
print("- PageValues: little correlation with observed columns, yet we know the missingness")
print("  depends on the hidden value itself. This is exactly why MNAR cannot be diagnosed")
print("  from the data alone — it needs domain knowledge.")

fig, axes = plt.subplots(1, 3, figsize=(13, 3))
for ax, c in zip(axes, ["ProductRelated_Duration", "BounceRates", "PageValues"]):
    grp = work.groupby(c + "_missing")["ProductRelated"].median()
    grp.plot(kind="bar", ax=ax, color=["#C9D4DB", "#A6752C"])
    ax.set_title(f"{c}\nmedian ProductRelated by missingness"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

### Compare imputation strategies

Five strategies on the same column, judged on two criteria: **how close the filled values are to
the truth**, and **what happened to the variance**.

In [ ]:
# ---- Compare strategies against the known truth -------------------------
from sklearn.experimental import enable_iterative_imputer   # noqa: F401
from sklearn.impute import SimpleImputer, KNNImputer, IterativeImputer

target_col   = "ProductRelated_Duration"      # the MCAR column
support_cols = ["Administrative", "Informational", "ProductRelated",
                "ExitRates", "SpecialDay"]

observed = work[target_col]
actual   = truth[target_col]
mask     = observed.isna().values

strategies = {
    "drop rows"     : None,
    "mean"          : SimpleImputer(strategy="mean"),
    "median"        : SimpleImputer(strategy="median"),
    "KNN (k=5)"     : KNNImputer(n_neighbors=5),
    "iterative"     : IterativeImputer(max_iter=10, random_state=RANDOM_STATE),
}

block   = work[[target_col] + support_cols]
results = []

for name, imp in strategies.items():
    if imp is None:
        kept = observed.dropna()
        results.append({"strategy": name, "n": len(kept), "mean": kept.mean(),
                        "std": kept.std(), "skew": kept.skew(),
                        "MAE_vs_truth": np.nan, "rows_lost": int(mask.sum())})
        continue
    filled = pd.DataFrame(imp.fit_transform(block), columns=block.columns)[target_col]
    mae = np.abs(filled.values[mask] - actual.values[mask]).mean()
    results.append({"strategy": name, "n": len(filled), "mean": filled.mean(),
                    "std": filled.std(), "skew": filled.skew(),
                    "MAE_vs_truth": mae, "rows_lost": 0})

comp = pd.DataFrame(results).set_index("strategy")
comp.loc["TRUE VALUES"] = [len(actual), actual.mean(), actual.std(),
                           actual.skew(), 0.0, 0]
print(comp.round(3).to_string())

### Analyze variance change

Mean imputation gives every missing row the identical value. Those rows then contribute **zero
deviation** from the mean, so the standard deviation must fall. The dataset looks more precise
than it is — confidence intervals narrow and correlations weaken.

In [ ]:
# ---- Quantify the damage ------------------------------------------------
true_std = actual.std()
change = pd.DataFrame({
    "std"        : comp["std"],
    "vs_truth_%" : ((comp["std"] - true_std) / true_std * 100),
    "MAE"        : comp["MAE_vs_truth"],
}).drop(index="TRUE VALUES")
print(change.round(2).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))

change["vs_truth_%"].plot(kind="bar", ax=axes[0],
                          color=["#8B9199", "#A6752C", "#A6752C", "#1F6F6B", "#1F6F6B"])
axes[0].axhline(0, color="black", lw=0.8)
axes[0].set_title("Change in standard deviation vs. the truth (%)"); axes[0].set_xlabel("")

sns.kdeplot(actual, ax=axes[1], label="true values", lw=2, color="black")
for name, imp in [("mean", SimpleImputer(strategy="mean")),
                  ("median", SimpleImputer(strategy="median")),
                  ("KNN (k=5)", KNNImputer(n_neighbors=5))]:
    f = pd.DataFrame(imp.fit_transform(block), columns=block.columns)[target_col]
    sns.kdeplot(f, ax=axes[1], label=name, lw=1.4)
axes[1].set_xlim(0, actual.quantile(0.97))
axes[1].legend(); axes[1].set_title("Distribution after imputation")
plt.tight_layout(); plt.show()

print("Interpretation")
print("- Mean and median imputation shrink the spread and add a spike at the fill value.")
print("- KNN and iterative imputation preserve the shape far better because they use the")
print("  other columns rather than a single constant.")
print("- 'Drop rows' preserves the distribution exactly but discards data — and is only")
print("  unbiased when the mechanism is genuinely MCAR.")

In [ ]:
# ---- The real-world case: disguised missingness in Pima ----------------
pima_path = find("pima.csv") or find("diabetes.csv")
if pima_path:
    pima = pd.read_csv(pima_path)
    zero_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
    zero_cols = [c for c in zero_cols if c in pima.columns]

    print("pandas reports nulls:", int(pima.isna().sum().sum()))
    disguised = pd.DataFrame({
        "zeros"  : (pima[zero_cols] == 0).sum(),
        "zeros_%": (pima[zero_cols] == 0).mean().mul(100).round(1),
    })
    print("\nValues recorded as zero — physiologically impossible:")
    print(disguised.to_string())

    p = pima.copy()
    p[zero_cols] = p[zero_cols].replace(0, np.nan)
    print("\nAfter replacing sentinels with NaN, true missingness:")
    print(p[zero_cols].isna().mean().mul(100).round(1).to_string())
    print(f"\nInsulin std: observed {p['Insulin'].std():.2f} -> "
          f"after mean imputation {p['Insulin'].fillna(p['Insulin'].mean()).std():.2f}")
else:
    print("pima.csv not found — skipping the disguised-missingness demonstration.")

### Deliverable — P17-18

A notebook containing the injected-missingness experiment, the strategy comparison table with
MAE against the truth, the variance-change chart, and a **three-sentence justification** of the
strategy you would adopt and the mechanism you are assuming.

---

# P19-20 · Outlier detection and treatment

**Week 10 · Module 3 · CO3**

**Objective.** Detect outliers by several methods, decide per variable whether to treat, keep or investigate, and measure the impact of that decision.


### Detect outliers

Three univariate rules and one multivariate method. They disagree, and the disagreement is
informative — a point flagged by every method deserves investigation; a point flagged by one
may be an artefact of that rule's assumptions.

In [ ]:
# ---- Detection rules ----------------------------------------------------
def iqr_flags(s, k=1.5):
    q1, q3 = s.quantile(.25), s.quantile(.75)
    iqr = q3 - q1
    return (s < q1 - k * iqr) | (s > q3 + k * iqr)

def z_flags(s, t=3.0):
    sd = s.std()
    return (s - s.mean()).abs() / sd > t if sd else pd.Series(False, index=s.index)

def mad_flags(s, t=3.5):
    med = s.median()
    mad = (s - med).abs().median()
    if mad == 0:                       # fall back when more than half the values are identical
        mad = (s - med).abs().mean() or 1.0
        return 0.7979 * (s - med).abs() / mad > t
    return 0.6745 * (s - med).abs() / mad > t

check = ["Administrative_Duration", "Informational_Duration", "ProductRelated",
         "ProductRelated_Duration", "BounceRates", "ExitRates", "PageValues"]

detect = pd.DataFrame({
    "IQR_%"      : {c: iqr_flags(df_clean[c]).mean() * 100 for c in check},
    "z_score_%"  : {c: z_flags(df_clean[c]).mean() * 100 for c in check},
    "mod_z_%"    : {c: mad_flags(df_clean[c]).mean() * 100 for c in check},
    "skew"       : {c: df_clean[c].skew() for c in check},
})
print(detect.round(2).to_string())

print("\nInterpretation")
print("- The three rules disagree because each assumes a different shape.")
print("- The IQR fence flags the tail of a skewed distribution by construction, so a high")
print("  IQR_% on a high-skew column is expected — it is not evidence of bad data.")
print("- z-score under-flags here: the extreme values inflate the standard deviation that")
print("  the rule itself depends on. This is masking.")

In [ ]:
# ---- Multivariate detection ---------------------------------------------
from sklearn.ensemble import IsolationForest

mv_cols = ["Administrative", "Informational", "ProductRelated",
           "ProductRelated_Duration", "BounceRates", "ExitRates", "PageValues"]

iso = IsolationForest(contamination=0.02, random_state=RANDOM_STATE, n_estimators=200)
iso_flag = iso.fit_predict(StandardScaler().fit_transform(df_clean[mv_cols])) == -1

uni_any = np.zeros(len(df_clean), dtype=bool)
for c in mv_cols:
    uni_any |= iqr_flags(df_clean[c]).values

overlap = pd.crosstab(pd.Series(uni_any, name="flagged by any IQR rule"),
                      pd.Series(iso_flag, name="flagged by IsolationForest"))
print(overlap.to_string())

only_mv = int((iso_flag & ~uni_any).sum())
print(f"\nRows flagged ONLY by the multivariate method: {only_mv}")
print("These are ordinary on every single axis but implausible in combination —")
print("for example a very long session that viewed almost no pages.")

print("\nProfile of the multivariate-only outliers (medians):")
print(pd.DataFrame({
    "multivariate-only": df_clean.loc[iso_flag & ~uni_any, mv_cols].median(),
    "everyone else"    : df_clean.loc[~(iso_flag & ~uni_any), mv_cols].median(),
}).round(2).to_string())

### Treatment — and comparing before/after impact

Four options: **investigate, remove, treat, keep.** Never delete silently. The question that
settles most cases is: *does the decision actually change the answer?*

In [ ]:
# ---- Does it change the conclusion? ------------------------------------
def impact(frame, keep_mask, label):
    sub = frame[keep_mask]
    return {
        "scenario"       : label,
        "rows"           : len(sub),
        "PageValues_mean": sub["PageValues"].mean(),
        "PRD_mean"       : sub["ProductRelated_Duration"].mean(),
        "PRD_median"     : sub["ProductRelated_Duration"].median(),
        "PRD_std"        : sub["ProductRelated_Duration"].std(),
        "corr_PV_target" : sub["PageValues"].corr(sub[TARGET].astype(int)),
        "conversion_%"   : sub[TARGET].mean() * 100,
    }

rows = [
    impact(df_clean, np.ones(len(df_clean), dtype=bool), "all rows"),
    impact(df_clean, ~uni_any,  "drop univariate IQR flags"),
    impact(df_clean, ~iso_flag, "drop IsolationForest flags"),
]
comparison = pd.DataFrame(rows).set_index("scenario")
print(comparison.round(3).to_string())

base = comparison.loc["all rows"]
delta = ((comparison - base) / base * 100).drop(index="all rows")
print("\nPercentage change against 'all rows'")
print(delta.round(2).to_string())

print("\nInterpretation")
print("- Dropping the IQR flags removes a large share of rows and moves the correlation")
print("  materially. That is not cleaning — it is changing the dataset to suit the method.")
print("- The conversion rate barely moves, which tells you the outliers are not driving")
print("  the headline finding.")

In [ ]:
# ---- Treatment options compared ----------------------------------------
col = "ProductRelated_Duration"
s   = df_clean[col]

lo, hi   = s.quantile(0.01), s.quantile(0.99)
winsor   = s.clip(lo, hi)
logged   = np.log1p(s)
q1, q3   = s.quantile(.25), s.quantile(.75)
removed  = s[~iqr_flags(s)]

treat = pd.DataFrame({
    "treatment": ["none", "winsorise 1-99%", "log1p", "remove IQR flags"],
    "n"        : [len(s), len(winsor), len(logged), len(removed)],
    "mean"     : [s.mean(), winsor.mean(), logged.mean(), removed.mean()],
    "median"   : [s.median(), winsor.median(), logged.median(), removed.median()],
    "std"      : [s.std(), winsor.std(), logged.std(), removed.std()],
    "skew"     : [s.skew(), winsor.skew(), logged.skew(), removed.skew()],
    "IQR_out_%": [iqr_flags(s).mean()*100, iqr_flags(winsor).mean()*100,
                  iqr_flags(logged).mean()*100, iqr_flags(removed).mean()*100],
}).set_index("treatment")
print(treat.round(3).to_string())

fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for ax, (name, data) in zip(axes, [("none", s), ("winsorised", winsor),
                                   ("log1p", logged), ("IQR removed", removed)]):
    sns.histplot(data, bins=50, ax=ax, color="#A6752C")
    ax.set_title(f"{name} — skew {data.skew():.2f}"); ax.set_xlabel("")
plt.tight_layout(); plt.show()

print("\nlog1p reduces skew dramatically while keeping every row. For a right-skewed")
print("positive variable it is almost always the better first move than deletion.")

In [ ]:
# ---- Per-variable decision table ---------------------------------------
decisions = pd.DataFrame([
    dict(variable="ProductRelated_Duration", flags=f"{iqr_flags(df_clean['ProductRelated_Duration']).sum():,}",
         decision="Transform (log1p)",
         reason="Right-skewed by nature; long sessions are genuine behaviour, not errors"),
    dict(variable="PageValues", flags=f"{iqr_flags(df_clean['PageValues']).sum():,}",
         decision="Keep",
         reason="The high values ARE the signal — they mark checkout-path sessions"),
    dict(variable="BounceRates", flags=f"{iqr_flags(df_clean['BounceRates']).sum():,}",
         decision="Keep",
         reason="Bounded in [0,1] and valid throughout; the spike at 0 is real behaviour"),
    dict(variable="Administrative_Duration", flags=f"{iqr_flags(df_clean['Administrative_Duration']).sum():,}",
         decision="Winsorise 99th",
         reason="Extreme tail is plausibly idle-time logging, not engagement"),
    dict(variable="multivariate-only rows", flags=str(only_mv),
         decision="Investigate",
         reason="Implausible combinations; check against the source before deciding"),
])
print(decisions.to_string(index=False))
print("\nWhatever you choose, the count and the justification go into the cleaning log.")

### Deliverable — P19-20

A notebook with the **per-variable decision table**, the before/after impact comparison, and
distribution plots for every variable you treated.

---

# P23-24 · Transformation and scaling

**Week 12 · Module 4 · CO4**

**Objective.** Apply transformations and scalers, and compare the distribution before and after each one instead of assuming it worked.


### Apply transformations

**Transformation** changes the *shape* of a distribution (non-linear: log, sqrt, Box-Cox).
**Scaling** changes its *location and spread* (linear: subtract a centre, divide by a spread).
Skew survives scaling untouched — so transform first if shape is the problem.

In [ ]:
# ---- Which columns need it? --------------------------------------------
from sklearn.preprocessing import (StandardScaler, MinMaxScaler,
                                   RobustScaler, PowerTransformer, QuantileTransformer)

model_cols = ["Administrative", "Administrative_Duration", "Informational",
              "Informational_Duration", "ProductRelated", "ProductRelated_Duration",
              "BounceRates", "ExitRates", "PageValues"]

skew_before = df_clean[model_cols].skew().sort_values(ascending=False)
print("Skew before transformation")
print(skew_before.round(3).to_string())
needs = skew_before[skew_before.abs() > 1].index.tolist()
print(f"\n{len(needs)} columns exceed |skew| = 1: {needs}")

In [ ]:
# ---- Compare transforms on one column, then measure --------------------
col = "ProductRelated_Duration"
s   = df_clean[col]

yj = PowerTransformer(method="yeo-johnson")            # handles zeros and negatives
qt = QuantileTransformer(output_distribution="normal", n_quantiles=1000,
                         random_state=RANDOM_STATE)

variants = {
    "original"      : s,
    "log1p"         : np.log1p(s),
    "sqrt"          : np.sqrt(s),
    "yeo-johnson"   : pd.Series(yj.fit_transform(s.to_frame()).ravel(), index=s.index),
    "quantile-normal": pd.Series(qt.fit_transform(s.to_frame()).ravel(), index=s.index),
}

table = pd.DataFrame({
    "skew"     : {k: v.skew() for k, v in variants.items()},
    "kurtosis" : {k: v.kurtosis() for k, v in variants.items()},
    "mean"     : {k: v.mean() for k, v in variants.items()},
    "std"      : {k: v.std() for k, v in variants.items()},
})
table["|skew| reduction %"] = (
    (abs(table.loc["original", "skew"]) - table["skew"].abs())
    / abs(table.loc["original", "skew"]) * 100)
print(table.round(3).to_string())

fig, axes = plt.subplots(1, 5, figsize=(15, 2.9))
for ax, (name, v) in zip(axes, variants.items()):
    sns.histplot(v, bins=50, ax=ax, color="#6B4C7A")
    ax.set_title(f"{name}\nskew {v.skew():.2f}", fontsize=9); ax.set_xlabel("")
plt.tight_layout(); plt.show()

print("\nApply, then RE-MEASURE. A transform that does not reduce skew has not helped.")

In [ ]:
# ---- Transform every skewed column and re-check -------------------------
df_t = df_clean.copy()
for c in needs:
    df_t[c] = np.log1p(df_t[c].clip(lower=0))

skew_after = df_t[model_cols].skew()
compare = pd.DataFrame({"before": skew_before, "after": skew_after.reindex(skew_before.index)})
compare["improved"] = compare["after"].abs() < compare["before"].abs()
print(compare.round(3).to_string())

plt.figure(figsize=(9, 3.4))
idx = np.arange(len(compare)); w = 0.38
plt.bar(idx - w/2, compare["before"], w, label="before", color="#8B9199")
plt.bar(idx + w/2, compare["after"],  w, label="after log1p", color="#6B4C7A")
plt.xticks(idx, compare.index, rotation=35, ha="right"); plt.axhline(0, color="k", lw=.8)
plt.ylabel("skew"); plt.legend(); plt.title("Skew before and after transformation")
plt.tight_layout(); plt.show()

### Apply scaling and compare before/after

In [ ]:
# ---- Four scalers on the same data --------------------------------------
scalers = {
    "StandardScaler": StandardScaler(),
    "MinMaxScaler"  : MinMaxScaler(),
    "RobustScaler"  : RobustScaler(),
}

rows = []
for name, sc in scalers.items():
    out = pd.DataFrame(sc.fit_transform(df_t[model_cols]), columns=model_cols)
    rows.append({"scaler": name, "mean": out.values.mean(), "std": out.values.std(),
                 "min": out.values.min(), "max": out.values.max(),
                 "skew(PRD)": out["ProductRelated_Duration"].skew()})
rows.insert(0, {"scaler": "none", "mean": df_t[model_cols].values.mean(),
                "std": df_t[model_cols].values.std(), "min": df_t[model_cols].values.min(),
                "max": df_t[model_cols].values.max(),
                "skew(PRD)": df_t["ProductRelated_Duration"].skew()})
print(pd.DataFrame(rows).set_index("scaler").round(3).to_string())

print("\nNote the last column: every scaler leaves the skew unchanged.")
print("Scaling moves and stretches a distribution; it never reshapes it.")

In [ ]:
# ---- Why MinMaxScaler is fragile ---------------------------------------
demo = df_t[["ProductRelated_Duration"]].copy()
demo_out = demo.copy()
demo_out.iloc[0, 0] = demo.iloc[:, 0].max() * 25      # one extreme value

fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
for ax, (title, frame) in zip(axes, [("clean", demo), ("one extreme value added", demo_out)]):
    for name, sc in [("MinMax", MinMaxScaler()), ("Robust", RobustScaler())]:
        v = sc.fit_transform(frame).ravel()
        sns.kdeplot(v, ax=ax, label=name, lw=1.6)
    ax.set_title(title); ax.legend(); ax.set_xlim(-2, 3)
plt.tight_layout(); plt.show()

for title, frame in [("clean", demo), ("with outlier", demo_out)]:
    mm = MinMaxScaler().fit_transform(frame).ravel()
    rb = RobustScaler().fit_transform(frame).ravel()
    print(f"{title:14} MinMax median {np.median(mm):.4f} | Robust median {np.median(rb):.4f}")

print("\nOne extreme value stretches the MinMax range so every ordinary value compresses")
print("toward zero. RobustScaler uses the median and IQR and is barely affected.")

### Deliverable — P23-24

A notebook with a **skew-before / skew-after table**, the scaler comparison, the MinMax fragility
demonstration, and a justified choice of transform and scaler for every column you changed.

---

# P25-26 · Feature engineering and selection

**Week 13 · Module 4 · CO4**

**Objective.** Construct features that encode domain knowledge, then compare filter, wrapper and embedded selection methods on the result.


### Feature engineering exercises

A feature is a question you have decided to ask of every row. Each constructed feature below
carries a one-line rationale — build nothing you cannot justify.

In [ ]:
# ---- Construct features -------------------------------------------------
fe = df_t.copy()

# 1 — total session depth and duration: overall engagement
fe["total_pages"]    = fe["Administrative"] + fe["Informational"] + fe["ProductRelated"]
fe["total_duration"] = (fe["Administrative_Duration"] + fe["Informational_Duration"]
                        + fe["ProductRelated_Duration"])

# 2 — average time per page: intent quality rather than raw volume
fe["avg_time_per_page"] = fe["total_duration"] / fe["total_pages"].replace(0, np.nan)
fe["avg_time_per_page"] = fe["avg_time_per_page"].fillna(0)

# 3 — product focus: what share of the session was spent on product pages
fe["product_focus"] = fe["ProductRelated"] / fe["total_pages"].replace(0, np.nan)
fe["product_focus"] = fe["product_focus"].fillna(0)

# 4 — engagement gap: exit rate relative to bounce rate
fe["exit_bounce_gap"] = fe["ExitRates"] - fe["BounceRates"]

# 5 — indicators: cheap, interpretable, often strong
fe["has_page_value"]   = (df_clean["PageValues"] > 0).astype(int)
fe["is_returning"]     = (fe["VisitorType"] == "Returning_Visitor").astype(int)
fe["is_holiday_month"] = fe["Month"].isin(["Nov", "Dec"]).astype(int)

# 6 — interaction: value only realised when the visitor engages
fe["value_x_depth"] = df_clean["PageValues"] * fe["ProductRelated"]

new_features = ["total_pages", "total_duration", "avg_time_per_page", "product_focus",
                "exit_bounce_gap", "has_page_value", "is_returning",
                "is_holiday_month", "value_x_depth"]

rationale = {
    "total_pages"      : "Overall session depth across all page types",
    "total_duration"   : "Overall time invested in the session",
    "avg_time_per_page": "Quality of attention rather than raw volume",
    "product_focus"    : "Share of the session spent on product pages",
    "exit_bounce_gap"  : "Sessions that continued after landing",
    "has_page_value"   : "Did the session reach a page on the checkout path",
    "is_returning"     : "Prior familiarity with the retailer",
    "is_holiday_month" : "November-December seasonal peak",
    "value_x_depth"    : "Page value only converts when the visitor browses",
}
print(pd.DataFrame({"feature": new_features,
                    "rationale": [rationale[f] for f in new_features]}).to_string(index=False))

print("\nAssociation of each new feature with the target:")
corrs = fe[new_features].corrwith(fe[TARGET].astype(int)).sort_values(key=abs, ascending=False)
print(corrs.round(3).to_string())

In [ ]:
# ---- Assemble the modelling matrix --------------------------------------
feature_cols = model_cols + new_features
Xf = fe[feature_cols].replace([np.inf, -np.inf], np.nan).fillna(0)
yf = fe[TARGET].astype(int)

print("Feature matrix:", Xf.shape)
print("Target balance :", f"{yf.mean()*100:.2f}% positive")

from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(
    Xf, yf, test_size=0.2, stratify=yf, random_state=RANDOM_STATE)
print(f"Train {X_tr.shape}  Test {X_te.shape}")
print("Selection is fitted on the TRAINING data only — see P29-30 on leakage.")

### Feature selection comparison

Three families, three different questions. **Agreement between them is the signal worth trusting.**

In [ ]:
# ---- Filter, wrapper, embedded ------------------------------------------
from sklearn.feature_selection import (SelectKBest, f_classif,
                                       mutual_info_classif, RFE)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

K = 8
scaler_sel = StandardScaler().fit(X_tr)
X_tr_s = scaler_sel.transform(X_tr)

# filter 1 — ANOVA F
anova = SelectKBest(f_classif, k=K).fit(X_tr_s, y_tr)
# filter 2 — mutual information (catches non-linear dependence)
mi = SelectKBest(mutual_info_classif, k=K).fit(X_tr_s, y_tr)
# wrapper — recursive feature elimination
rfe = RFE(LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
          n_features_to_select=K).fit(X_tr_s, y_tr)
# embedded 1 — random forest importance
rf = RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE,
                            n_jobs=-1).fit(X_tr, y_tr)
# embedded 2 — L1 logistic regression
lasso = LogisticRegression(penalty="l1", solver="liblinear", C=0.1,
                           max_iter=2000, random_state=RANDOM_STATE).fit(X_tr_s, y_tr)

sel = pd.DataFrame({
    "anova_F"   : anova.scores_,
    "mutual_info": mi.scores_,
    "rf_importance": rf.feature_importances_,
    "lasso_coef": np.abs(lasso.coef_.ravel()),
    "picked_anova": anova.get_support(),
    "picked_mi"   : mi.get_support(),
    "picked_rfe"  : rfe.support_,
}, index=feature_cols)

sel["picked_rf"]    = sel["rf_importance"].rank(ascending=False) <= K
sel["picked_lasso"] = sel["lasso_coef"] > 0
sel["votes"] = sel[["picked_anova", "picked_mi", "picked_rfe",
                    "picked_rf", "picked_lasso"]].sum(axis=1)

print(sel.sort_values("votes", ascending=False).round(4).to_string())

In [ ]:
# ---- Where do the methods agree? ---------------------------------------
consensus = sel[sel["votes"] >= 4].index.tolist()
contested = sel[(sel["votes"] > 0) & (sel["votes"] < 3)].index.tolist()

print(f"Chosen by 4 or 5 of 5 methods ({len(consensus)}):")
for f in consensus: print("   -", f)
print(f"\nChosen by only one or two methods ({len(contested)}) — treat with suspicion:")
for f in contested: print("   -", f)

plt.figure(figsize=(9, 4.4))
sns.barplot(x=sel["votes"].sort_values(ascending=False),
            y=sel["votes"].sort_values(ascending=False).index, color="#6B4C7A")
plt.xlabel("number of selection methods that chose the feature"); plt.ylabel("")
plt.title("Agreement across filter, wrapper and embedded selection")
plt.tight_layout(); plt.show()

engineered_kept = [f for f in consensus if f in new_features]
print(f"\n{len(engineered_kept)} of {len(new_features)} engineered features survived selection:")
print("  ", engineered_kept)
print("\nThat is the test of feature engineering — did the new features earn their place?")

In [ ]:
# ---- Does selection actually help? --------------------------------------
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def score_subset(cols, label):
    pipe = Pipeline([("scale", StandardScaler()),
                     ("model", LogisticRegression(max_iter=2000,
                                                  random_state=RANDOM_STATE))])
    s = cross_val_score(pipe, X_tr[cols], y_tr, cv=cv, scoring="roc_auc", n_jobs=-1)
    return {"feature set": label, "n_features": len(cols),
            "roc_auc_mean": s.mean(), "roc_auc_std": s.std()}

results = pd.DataFrame([
    score_subset(feature_cols, "all features"),
    score_subset(model_cols, "original only"),
    score_subset(consensus, "consensus selection"),
    score_subset(sel["rf_importance"].nlargest(5).index.tolist(), "top 5 by RF importance"),
]).set_index("feature set")
print(results.round(4).to_string())

print("\nInterpretation")
print("- A much smaller feature set that performs comparably is the better model:")
print("  it is cheaper, more stable and easier to explain.")
print("- Report the spread as well as the mean. A high mean with a wide spread is unstable.")

### Deliverable — P25-26

A notebook with the **engineered-feature table and rationales**, the five-method comparison, the
agreement chart, and a shortlist justified by method agreement rather than by a single score.

---

# P27-28 · PCA — implementation and interpretation

**Week 14 · Module 4 · CO4**

**Objective.** Implement PCA, visualise the components, and interpret the variance explained and the loadings.


### PCA implementation

PCA finds orthogonal directions of maximum variance. It is **unsupervised** — it never looks at
the target — and it is **scale-sensitive**, so standardising first is not optional.

In [ ]:
# ---- Fit PCA ------------------------------------------------------------
from sklearn.decomposition import PCA

pca_cols = model_cols                  # the transformed original variables
Xp = df_t[pca_cols]

Xp_s = StandardScaler().fit_transform(Xp)
pca  = PCA(random_state=RANDOM_STATE).fit(Xp_s)

evr = pca.explained_variance_ratio_ * 100
variance = pd.DataFrame({
    "component"      : [f"PC{i+1}" for i in range(len(evr))],
    "eigenvalue"     : pca.explained_variance_,
    "variance_%"     : evr,
    "cumulative_%"   : np.cumsum(evr),
}).set_index("component")
print(variance.round(2).to_string())

for target_var in (80, 90, 95):
    k = int(np.argmax(np.cumsum(evr) >= target_var) + 1)
    print(f"Components needed for {target_var}% of variance: {k}")
print("Components with eigenvalue > 1 (Kaiser rule):",
      int((pca.explained_variance_ > 1).sum()))

### Interpret variance explained

In [ ]:
# ---- Scree and cumulative variance --------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(12, 3.6))

axes[0].bar(range(1, len(evr) + 1), evr, color="#6B4C7A")
axes[0].plot(range(1, len(evr) + 1), evr, marker="o", color="#B5432E")
axes[0].set_title("Scree plot"); axes[0].set_xlabel("component")
axes[0].set_ylabel("variance explained (%)")

axes[1].plot(range(1, len(evr) + 1), np.cumsum(evr), marker="o", color="#6B4C7A")
for lvl, c in [(80, "#B5432E"), (95, "#8B9199")]:
    axes[1].axhline(lvl, ls="--", lw=1, color=c)
    axes[1].text(0.4, lvl + 1.2, f"{lvl}%", color=c, fontsize=9)
axes[1].set_title("Cumulative variance"); axes[1].set_xlabel("components")
axes[1].set_ylim(0, 105)
plt.tight_layout(); plt.show()

k80 = int(np.argmax(np.cumsum(evr) >= 80) + 1)
print(f"Retention decision: keep {k80} components for 80% of the variance "
      f"({len(pca_cols)} -> {k80}, a {100*(1-k80/len(pca_cols)):.0f}% reduction).")
print("A flat scree means the variables were already fairly independent — that is a finding,")
print("not a failure. Report it rather than forcing a two-component solution.")

### Visualize components

In [ ]:
# ---- Project and plot ---------------------------------------------------
P = pca.transform(Xp_s)
samp = np.random.RandomState(RANDOM_STATE).choice(len(P), 4000, replace=False)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

sns.scatterplot(x=P[samp, 0], y=P[samp, 1], hue=df_t[TARGET].values[samp],
                palette=["#C9D4DB", "#B5432E"], s=10, alpha=0.55, ax=axes[0])
axes[0].set_title("PC1 vs PC2, coloured by outcome")

sns.scatterplot(x=P[samp, 0], y=P[samp, 2], hue=df_t[TARGET].values[samp],
                palette=["#C9D4DB", "#B5432E"], s=10, alpha=0.55, ax=axes[1], legend=False)
axes[1].set_title("PC1 vs PC3")

# biplot: how each original variable loads onto the first two components
load = pca.components_[:2].T * np.sqrt(pca.explained_variance_[:2])
for i, name in enumerate(pca_cols):
    axes[2].arrow(0, 0, load[i, 0], load[i, 1], head_width=0.03,
                  color="#6B4C7A", alpha=0.8)
    axes[2].text(load[i, 0] * 1.12, load[i, 1] * 1.12, name, fontsize=7, ha="center")
axes[2].set_xlim(-1.2, 1.2); axes[2].set_ylim(-1.2, 1.2)
axes[2].axhline(0, lw=.6, color="grey"); axes[2].axvline(0, lw=.6, color="grey")
axes[2].set_title("Loading biplot (PC1 / PC2)")

for a in axes[:2]:
    a.set_xlabel(f"PC1 ({evr[0]:.1f}%)"); a.set_ylabel(f"PC2 ({evr[1]:.1f}%)")
plt.tight_layout(); plt.show()

In [ ]:
# ---- Loadings: what is each component actually made of? ----------------
loadings = pd.DataFrame(
    pca.components_[:4].T,
    columns=[f"PC{i+1}" for i in range(4)],
    index=pca_cols)

print("Loadings (first four components)")
print(loadings.round(3).to_string())

print("\nDominant variables per component")
for pc in loadings.columns:
    top = loadings[pc].abs().sort_values(ascending=False).head(3)
    parts = [f"{name} ({loadings.loc[name, pc]:+.2f})" for name in top.index]
    print(f"  {pc} ({evr[int(pc[2:])-1]:.1f}% var): " + ", ".join(parts))

plt.figure(figsize=(8, 3.6))
sns.heatmap(loadings, annot=True, fmt=".2f", center=0, cmap="PuOr",
            cbar_kws={"shrink": .8}, annot_kws={"size": 8})
plt.title("Component loadings"); plt.tight_layout(); plt.show()

print("\nNow NAME each component from its loadings — 'session depth', 'bounce/exit behaviour',")
print("'checkout intent'. A component you cannot name should not appear in your report.")

In [ ]:
# ---- Is the reduction worth it? ----------------------------------------
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

pipe_full = Pipeline([("scale", StandardScaler()),
                      ("model", LogisticRegression(max_iter=2000,
                                                   random_state=RANDOM_STATE))])
pipe_pca  = Pipeline([("scale", StandardScaler()),
                      ("pca", PCA(n_components=k80, random_state=RANDOM_STATE)),
                      ("model", LogisticRegression(max_iter=2000,
                                                   random_state=RANDOM_STATE))])

yv = df_t[TARGET].astype(int)
s_full = cross_val_score(pipe_full, Xp, yv, cv=cv, scoring="roc_auc", n_jobs=-1)
s_pca  = cross_val_score(pipe_pca,  Xp, yv, cv=cv, scoring="roc_auc", n_jobs=-1)

print(pd.DataFrame({
    "pipeline"  : [f"all {len(pca_cols)} variables", f"PCA -> {k80} components"],
    "roc_auc"   : [s_full.mean(), s_pca.mean()],
    "std"       : [s_full.std(), s_pca.std()],
}).round(4).to_string(index=False))

print("\nNote that PCA sits INSIDE the pipeline, so it is refitted on each training fold.")
print("Fitting PCA on the whole dataset before cross-validation would leak test information.")

### Deliverable — P27-28

A notebook containing the variance table, scree and cumulative plots, the retention decision with
its justification, the 2-D projections, the loading heatmap, and a **plain-language name for every
retained component**.

---

# P29-30 · Full pipeline — industry-style notebook

**Week 15 · Module 4 · CO1-CO4**

**Objective.** Assemble every step into one leak-free pipeline, validate it honestly, and produce the report structure expected in the final project.


### Full pipeline implementation

Everything so far, assembled in the correct order. The single governing rule:

> **Split first. Fit every transformer on the training data only.**

A `Pipeline` enforces this automatically — which is why it is the required structure for the
final project.

In [ ]:
# ---- Start again from the raw file, so the pipeline is self-contained ---
raw = pd.read_csv(find("online_shoppers_intention.csv")).drop_duplicates().reset_index(drop=True)
print("Raw, deduplicated:", raw.shape)

TARGET = "Revenue"
y_all = raw[TARGET].astype(int)
X_all = raw.drop(columns=[TARGET])

# --- feature construction expressed as a reusable function -----------------
def engineer(frame):
    f = frame.copy()
    f["total_pages"]    = f["Administrative"] + f["Informational"] + f["ProductRelated"]
    f["total_duration"] = (f["Administrative_Duration"] + f["Informational_Duration"]
                           + f["ProductRelated_Duration"])
    f["avg_time_per_page"] = (f["total_duration"] / f["total_pages"].replace(0, np.nan)).fillna(0)
    f["product_focus"]     = (f["ProductRelated"] / f["total_pages"].replace(0, np.nan)).fillna(0)
    f["exit_bounce_gap"]   = f["ExitRates"] - f["BounceRates"]
    f["has_page_value"]    = (f["PageValues"] > 0).astype(int)
    f["is_holiday_month"]  = f["Month"].isin(["Nov", "Dec"]).astype(int)
    return f

X_all = engineer(X_all)
print("After feature construction:", X_all.shape)

In [ ]:
# ---- Split BEFORE anything is fitted ------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, stratify=y_all, random_state=RANDOM_STATE)

print(f"Train {X_train.shape}   Test {X_test.shape}")
print(f"Positive rate — train {y_train.mean()*100:.2f}%  test {y_test.mean()*100:.2f}%")
print("Stratified, so both sides carry the same class balance.")

In [ ]:
# ---- Build the preprocessing + model pipeline ---------------------------
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

numeric_features = X_train.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = ["Month", "VisitorType", "OperatingSystems",
                        "Browser", "Region", "TrafficType", "Weekend"]
categorical_features = [c for c in categorical_features if c in X_train.columns]

numeric_pipe = Pipeline([
    ("impute", SimpleImputer(strategy="median")),
    ("log",    FunctionTransformer(lambda a: np.log1p(np.clip(a, 0, None)),
                                   feature_names_out="one-to-one")),
    ("scale",  StandardScaler()),
])

categorical_pipe = Pipeline([
    # Cast to plain strings first: these columns mix integer codes with text labels,
    # and an imputer will otherwise try to read 'Dec' as a number.
    ("as_text", FunctionTransformer(lambda d: pd.DataFrame(d).astype(str),
                                    feature_names_out="one-to-one")),
    ("impute",  SimpleImputer(strategy="most_frequent")),
    ("onehot",  OneHotEncoder(handle_unknown="ignore", drop="first",
                              min_frequency=0.01, sparse_output=False)),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipe, numeric_features),
    ("cat", categorical_pipe, categorical_features),
], remainder="drop")

pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=3000, class_weight="balanced",
                                 random_state=RANDOM_STATE)),
])

print(f"Numeric features    : {len(numeric_features)}")
print(f"Categorical features: {len(categorical_features)}")
print("\nEvery fitted step lives inside the pipeline. Nothing is fitted before the split.")

In [ ]:
# ---- Cross-validate on the training data only ---------------------------
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_auc_score, average_precision_score,
                             RocCurveDisplay, PrecisionRecallDisplay)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

for metric in ["roc_auc", "average_precision", "f1"]:
    s = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring=metric, n_jobs=-1)
    print(f"{metric:>18}: {s.mean():.4f} +/- {s.std():.4f}")

print("\nAccuracy is deliberately absent: at 15% positives it rewards predicting 'never'.")
print("ROC-AUC and average precision (PR-AUC) are the honest choices here.")

In [ ]:
# ---- Compare against a second model, then evaluate ONCE on the test set -
rf_pipeline = Pipeline([
    ("preprocess", preprocess),
    ("model", RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                     random_state=RANDOM_STATE, n_jobs=-1)),
])

for name, p in [("logistic regression", pipeline), ("random forest", rf_pipeline)]:
    s = cross_val_score(p, X_train, y_train, cv=cv, scoring="roc_auc", n_jobs=-1)
    print(f"{name:>22}: CV ROC-AUC {s.mean():.4f} +/- {s.std():.4f}")

# the test set is touched exactly once, at the very end
final = rf_pipeline.fit(X_train, y_train)
proba = final.predict_proba(X_test)[:, 1]
pred  = final.predict(X_test)

print(f"\nHELD-OUT TEST SET (used once)")
print(f"  ROC-AUC          : {roc_auc_score(y_test, proba):.4f}")
print(f"  Average precision: {average_precision_score(y_test, proba):.4f}")
print("\n", classification_report(y_test, pred, target_names=["no purchase", "purchase"]))

fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
RocCurveDisplay.from_predictions(y_test, proba, ax=axes[0], color="#6B4C7A")
axes[0].set_title("ROC curve")
PrecisionRecallDisplay.from_predictions(y_test, proba, ax=axes[1], color="#6B4C7A")
axes[1].set_title("Precision-recall curve")
sns.heatmap(confusion_matrix(y_test, pred), annot=True, fmt=",d", cmap="Purples",
            ax=axes[2], cbar=False,
            xticklabels=["pred no", "pred yes"], yticklabels=["true no", "true yes"])
axes[2].set_title("Confusion matrix")
plt.tight_layout(); plt.show()

### Leakage audit

Four paths, each checked explicitly. **Document this check in your final project.**

In [ ]:
# ---- Leakage audit ------------------------------------------------------
audit_rows = [
    dict(path="1. Preprocessing fitted on all data",
         risk="Scaler/imputer sees test statistics",
         status="CLOSED",
         evidence="All transformers live inside the Pipeline; fitted per fold on train only"),
    dict(path="2. Target leakage",
         risk="A feature encodes the answer",
         status="CHECKED",
         evidence="All features derive from session behaviour observable before checkout"),
    dict(path="3. Temporal leakage",
         risk="Training on rows recorded after test rows",
         status="NOTED",
         evidence="Month is available but no row-level timestamp; a random split is used. "
                  "With timestamps, a time-based split would be required"),
    dict(path="4. Duplicate leakage",
         risk="Same entity in train and test",
         status="CLOSED",
         evidence="drop_duplicates() applied before the split"),
]
print(pd.DataFrame(audit_rows).to_string(index=False))

# a blunt but effective check: is any single feature implausibly predictive?
single = {}
for c in numeric_features:
    try:
        single[c] = roc_auc_score(y_train, X_train[c])
    except Exception:
        pass
suspect = pd.Series(single).sort_values(ascending=False).head(5)
print("\nSingle-feature ROC-AUC (top 5) — anything above ~0.95 suggests leakage:")
print(suspect.round(4).to_string())
print("Highest is well below that threshold, so no feature is standing in for the target.")

### Industry-style notebook submission

The structure below is the required shape of the final project report. Each section answers one
question a reader will actually ask.

In [ ]:
# ---- Final report skeleton ----------------------------------------------
report = f"""
================================================================================
DATA209 FINAL PROJECT — EXPLORATORY DATA ANALYSIS REPORT
Dataset: Online Shoppers Purchasing Intention (UCI) | {raw.shape[0]:,} sessions x {raw.shape[1]} columns
================================================================================

1. WHAT THE DATA IS
   Source          : UCI Machine Learning Repository
   Unit of analysis: one browsing session
   Sampling frame  : sessions on a single retailer over a 12-month period
   Period          : {sorted(raw['Month'].unique())}
   Target          : Revenue — {y_all.mean()*100:.1f}% positive
   Cannot speak to : other retailers, other periods, or offline behaviour

2. WHAT WAS WRONG WITH IT
   Duplicates removed  : {12330 - len(raw):,} exact duplicate rows
   Explicit nulls      : {raw.isna().sum().sum()}
   Disguised missing   : none found in this dataset (contrast with pima.csv)
   Strongly skewed     : {len(needs)} of {len(model_cols)} numeric columns, log1p applied
   Rare categories     : levels below 1% grouped into 'Other' at encoding
   Validity violations : none against the domain rules in P15-16

3. WHAT IT SHOWS
   - PageValues is the strongest single separator of converting sessions
   - Returning visitors convert at a different rate from new visitors
   - Conversion is seasonal, rising into the November-December period
   - BounceRates and ExitRates are near-duplicates (multicollinear); keep one
   - K-means finds no strong natural segmentation (silhouette {overall:.2f})

4. WHAT COMES NEXT
   H1  Sessions reaching a positive-PageValue page convert far more often
       falsified by: no difference once session depth is controlled for
   H2  Returning visitors convert more because of prior evaluation
       falsified by: difference vanishes after conditioning on month and traffic type
   H3  Conversion peaks in the holiday period
       falsified by: monthly rates within sampling variation of the overall rate

   Modelling-ready dataset: {len(numeric_features)} numeric + {len(categorical_features)} categorical
   Recommended next step  : threshold-tuned classifier optimised for precision at
                            fixed recall, since the cost of a wasted contact is low
                            and the cost of a missed purchase is high

5. LIMITATIONS
   - No user identifier, so repeat visitors cannot be linked across sessions
   - No row-level timestamp, so a strict temporal split is impossible
   - Promotional calendar is unobserved and confounds the seasonal finding
================================================================================
"""
print(report)

### Deliverable — P29-30 and the final project

An industry-style notebook plus a 12-minute presentation, worth 30% of the course.

**Submission checklist**

- [ ] Kernel restarted and *Run All* completes without error
- [ ] Relative paths only — no `C:\Users\...`
- [ ] `random_state` set everywhere it exists
- [ ] Split occurs before any transformer is fitted
- [ ] All preprocessing inside a `Pipeline`
- [ ] Test set evaluated exactly once
- [ ] Leakage audit table included
- [ ] Metric appropriate to the class imbalance — not accuracy
- [ ] Every figure carries a sentence saying what it shows
- [ ] Limitations section states what the data cannot support